In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("../src")

In [3]:
PDF_PATH = pdf_path = "../data/XQuality/Textual/Chapitre_8_Alarmes_et_messages.pdf"
MODEL_NAME = model_name = "openai/gpt-oss-20b"

SEED_ONTOLOGY_INPUT = str("../data/ontology/ContextOntology-COInd4.owl")
ONTOLOGY_PATH = "../data/ontology/ContextOntology-COInd4.owl"

# SinglePassLLM

In [22]:
# Ground truth
GT_JSON_PATH = r"../../data/XQuality/Examples/XQuality_all_triplets_flat_en.json"

# SinglePass parsed JSON produced by your baseline
SINGLEPASS_PARSED_JSON_PATH = r"./outputs_singlepass_xquality/Chapitre_8_Alarmes_et_messages__singlepass_parsed.json"

# Optional ontology path used only for soft ontology conformance checks
SINGLEPASS_ONTOLOGY_PATH = r"../../data/ontology/ContextOntology-COInd4.owl"

In [26]:
# ============================================================
# STRICT RELATION EVALUATION FOR SINGLEPASS vs XQUALITY GT
# - alarm anchoring only
# - NO relation inversion tolerance
# - NO CAUSES <-> TRIGGERS flexible matching
# - stricter entity matching
# - stricter tail thresholds
# ============================================================

from rapidfuzz import fuzz
from collections import defaultdict
import json
import re

# ============================================================
# CONFIG
# ============================================================

ENTITY_SIM_THRESHOLD = 92

REL_HEAD_SIM_THRESHOLD = 94
REL_TAIL_SIM_THRESHOLD_DEFAULT = 94
REL_TAIL_SIM_THRESHOLD_REQUIRES = 92

PRINT_DEBUG_EXAMPLES = True
MAX_DEBUG_MATCHES = 20

# ============================================================
# HELPERS
# ============================================================

def normalize_text(text: str) -> str:
    """Normalize text conservatively for matching."""
    if text is None:
        return ""
    text = str(text).lower().strip()
    text = re.sub(r"[_/\\:;|]+", " ", text)
    text = text.replace("-", " ")
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def safe_div(a: float, b: float) -> float:
    """Safe division."""
    return a / b if b != 0 else 0.0


def harmonic_f1(p: float, r: float) -> float:
    """F1 from precision and recall."""
    return (2 * p * r / (p + r)) if (p + r) != 0 else 0.0


def extract_alarm_number(text: str):
    """Extract alarm number from strings like 'Alarm 1001'."""
    if not text:
        return None
    txt = normalize_text(text)
    m = re.search(r"\balarm\s+(\d{3,5})\b", txt)
    if m:
        return m.group(1)
    return None


def relation_tail_threshold(rel: str) -> float:
    """Relation-specific tail threshold."""
    rel_n = normalize_text(rel)
    if rel_n == "requires":
        return REL_TAIL_SIM_THRESHOLD_REQUIRES
    return REL_TAIL_SIM_THRESHOLD_DEFAULT


def is_generic_entity(text: str) -> bool:
    """
    Reject very generic labels that create false matches.
    """
    t = normalize_text(text)
    generic = {
        "operator",
        "maintenance",
        "maintenance technician",
        "technician",
        "officer",
        "person",
        "people",
        "device",
        "machine",
        "alarm",
        "check",
        "manual",
        "reference",
        "page",
        "diagram",
        "input",
        "cause",
        "intervention",
        "responsible",
        "actor",
    }
    return t in generic


def entity_similarity(a: str, b: str) -> float:
    """
    Stricter entity similarity than token_set_ratio.
    """
    a_n = normalize_text(a)
    b_n = normalize_text(b)

    if not a_n or not b_n:
        return 0.0

    if a_n == b_n:
        return 100.0

    # Reject generic entities early.
    if is_generic_entity(a_n) or is_generic_entity(b_n):
        return 0.0

    # Penalize large length mismatch.
    len_ratio = min(len(a_n), len(b_n)) / max(len(a_n), len(b_n))
    if len_ratio < 0.75:
        return 0.0

    # Penalize very different token counts.
    a_tok = a_n.split()
    b_tok = b_n.split()
    tok_ratio = min(len(a_tok), len(b_tok)) / max(len(a_tok), len(b_tok))
    if tok_ratio < 0.60:
        return 0.0

    s1 = fuzz.ratio(a_n, b_n)
    s2 = fuzz.token_sort_ratio(a_n, b_n)
    s3 = fuzz.token_set_ratio(a_n, b_n)

    # Conservative aggregate.
    return min(s1, s2, s3)


def relation_similarity(a: str, b: str) -> float:
    """
    Strict relation endpoint similarity.
    """
    a_n = normalize_text(a)
    b_n = normalize_text(b)

    if not a_n or not b_n:
        return 0.0

    if a_n == b_n:
        return 100.0

    # Reject generic endpoint labels.
    if is_generic_entity(a_n) or is_generic_entity(b_n):
        return 0.0

    len_ratio = min(len(a_n), len(b_n)) / max(len(a_n), len(b_n))
    if len_ratio < 0.75:
        return 0.0

    s1 = fuzz.ratio(a_n, b_n)
    s2 = fuzz.token_sort_ratio(a_n, b_n)

    # Strict, no token_set shortcut here.
    return min(s1, s2)


# ============================================================
# LOAD GT
# ============================================================

with open(GT_JSON_PATH, "r", encoding="utf-8") as f:
    gt_raw = json.load(f)

gt_triples = []
gt_entities = set()
alarm_no_to_label = {}

for row in gt_raw:
    node1 = str(row.get("Node 1", "")).strip()
    rel = str(row.get("RELATION", "")).strip().upper()
    node2 = str(row.get("Node 2", "")).strip()
    alarm_no = str(row.get("Alarm No.", "")).strip()

    if node1:
        gt_entities.add(node1)
    if node2:
        gt_entities.add(node2)

    # Alarm anchoring only.
    if alarm_no:
        if rel == "TRIGGERS" and node2:
            alarm_no_to_label[alarm_no] = node2
        elif rel in {"CAUSES", "REQUIRES", "HANDLED_BY", "REFERENCES"} and node1:
            alarm_no_to_label[alarm_no] = node1

    gt_triples.append({
        "head": node1,
        "rel": rel,
        "tail": node2,
        "alarm_no": alarm_no,
    })

print(f"Loaded GT triples: {len(gt_triples)}")
print(f"Loaded GT unique entities: {len(gt_entities)}")
print(f"Loaded GT alarm mappings: {len(alarm_no_to_label)}")

# ============================================================
# LOAD SINGLEPASS PARSED JSON
# ============================================================

with open(SINGLEPASS_PARSED_JSON_PATH, "r", encoding="utf-8") as f:
    singlepass_data = json.load(f)

raw_entities = singlepass_data.get("entities", [])
raw_relations = singlepass_data.get("relations", [])

print(f"Loaded SinglePass entities: {len(raw_entities)}")
print(f"Loaded SinglePass relations: {len(raw_relations)}")

# ============================================================
# BUILD SINGLEPASS ENTITY MAP
# ============================================================

entity_id_to_label = {}
for ent in raw_entities:
    ent_id = str(ent.get("id", "")).strip()
    ent_label = str(ent.get("label", "")).strip()
    if ent_id and ent_label:
        entity_id_to_label[ent_id] = ent_label

# ============================================================
# CANONICALIZE SINGLEPASS TRIPLES
# ============================================================

def canonicalize_alarm_label(label: str) -> str:
    """
    Replace 'Alarm 1001' with GT canonical alarm label when possible.
    """
    if not label:
        return label
    alarm_no = extract_alarm_number(label)
    if alarm_no and alarm_no in alarm_no_to_label:
        return alarm_no_to_label[alarm_no]
    return label


pred_triples = []
pred_entities = set()

for rel_item in raw_relations:
    head = str(rel_item.get("head", "")).strip()
    rel = str(rel_item.get("relation", "")).strip().upper()
    tail = str(rel_item.get("tail", "")).strip()
    evidence = str(rel_item.get("evidence", "")).strip()

    # Resolve IDs if present.
    if head in entity_id_to_label:
        head = entity_id_to_label[head]
    if tail in entity_id_to_label:
        tail = entity_id_to_label[tail]

    # Alarm anchoring only.
    head = canonicalize_alarm_label(head)
    tail = canonicalize_alarm_label(tail)

    if head:
        pred_entities.add(head)
    if tail:
        pred_entities.add(tail)

    pred_triples.append({
        "head": head,
        "rel": rel,
        "tail": tail,
        "evidence": evidence,
    })

print(f"Predicted entities for evaluation: {len(pred_entities)}")
print(f"Predicted triples for evaluation: {len(pred_triples)}")

# ============================================================
# ENTITY EVALUATION
# ============================================================

def greedy_entity_matching(pred_entities, gt_entities, threshold=ENTITY_SIM_THRESHOLD):
    """
    Strict one-to-one greedy entity matching.
    """
    pred_list = list(pred_entities)
    gt_list = list(gt_entities)

    candidates = []

    for i, p_ent in enumerate(pred_list):
        if is_generic_entity(p_ent):
            continue
        for j, g_ent in enumerate(gt_list):
            if is_generic_entity(g_ent):
                continue
            score = entity_similarity(p_ent, g_ent)
            if score >= threshold:
                candidates.append((score, i, j))

    candidates.sort(reverse=True, key=lambda x: (x[0], -x[1], -x[2]))

    matched_pred = set()
    matched_gt = set()
    matches = []

    for score, i, j in candidates:
        if i in matched_pred or j in matched_gt:
            continue
        matched_pred.add(i)
        matched_gt.add(j)
        matches.append({
            "pred": pred_list[i],
            "gt": gt_list[j],
            "score": score,
        })

    tp = len(matches)
    fp = len(pred_list) - tp
    fn = len(gt_list) - tp

    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    f1 = harmonic_f1(precision, recall)

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "matches": matches,
        "pred_list": pred_list,
        "gt_list": gt_list,
    }


entity_results = greedy_entity_matching(
    pred_entities=pred_entities,
    gt_entities=gt_entities,
    threshold=ENTITY_SIM_THRESHOLD,
)

print("\n================ ENTITY EVALUATION ================")
print(f"TP: {entity_results['tp']}")
print(f"FP: {entity_results['fp']}")
print(f"FN: {entity_results['fn']}")
print(f"Precision: {entity_results['precision']:.4f}")
print(f"Recall:    {entity_results['recall']:.4f}")
print(f"F1:        {entity_results['f1']:.4f}")

if PRINT_DEBUG_EXAMPLES:
    print("\nSample entity matches:")
    for m in entity_results["matches"][:MAX_DEBUG_MATCHES]:
        print(f"  PRED: {m['pred']}  <-->  GT: {m['gt']}   [score={m['score']:.1f}]")

# ============================================================
# STRICT RELATION MATCHING
# ============================================================

def triple_pair_score(pred, gt):
    """
    Strict triple score:
    - exact relation label
    - direct direction only
    - no CAUSES/TRIGGERS flexibility
    - strict endpoint matching
    """
    if pred["rel"] != gt["rel"]:
        return None

    head_score = relation_similarity(pred["head"], gt["head"])
    tail_score = relation_similarity(pred["tail"], gt["tail"])

    if head_score < REL_HEAD_SIM_THRESHOLD:
        return None
    if tail_score < relation_tail_threshold(pred["rel"]):
        return None

    total_score = 0.5 * head_score + 0.5 * tail_score

    return {
        "score": total_score,
        "head_score": head_score,
        "tail_score": tail_score,
        "direction": "direct",
        "gt_rel": gt["rel"],
    }


def greedy_relation_matching_strict(pred_triples, gt_triples):
    """
    Strict one-to-one greedy relation matching.
    """
    candidates = []

    for i, p in enumerate(pred_triples):
        for j, g in enumerate(gt_triples):
            cand = triple_pair_score(p, g)
            if cand is not None:
                candidates.append((
                    cand["score"],
                    cand["head_score"],
                    cand["tail_score"],
                    i,
                    j,
                    cand["direction"],
                    cand["gt_rel"],
                ))

    candidates.sort(reverse=True, key=lambda x: (x[0], x[1], x[2], -x[3], -x[4]))

    matched_pred = set()
    matched_gt = set()
    matches = []

    for total_score, head_score, tail_score, i, j, direction, gt_rel in candidates:
        if i in matched_pred or j in matched_gt:
            continue
        matched_pred.add(i)
        matched_gt.add(j)
        matches.append({
            "pred_idx": i,
            "gt_idx": j,
            "score": total_score,
            "head_score": head_score,
            "tail_score": tail_score,
            "direction": direction,
            "matched_gt_rel": gt_rel,
            "pred": pred_triples[i],
            "gt": gt_triples[j],
        })

    tp = len(matches)
    fp = len(pred_triples) - tp
    fn = len(gt_triples) - tp

    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    f1 = harmonic_f1(precision, recall)

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "matches": matches,
    }


relation_results = greedy_relation_matching_strict(pred_triples, gt_triples)

print("\n================ RELATION EVALUATION ================")
print(f"TP: {relation_results['tp']}")
print(f"FP: {relation_results['fp']}")
print(f"FN: {relation_results['fn']}")
print(f"Precision: {relation_results['precision']:.4f}")
print(f"Recall:    {relation_results['recall']:.4f}")
print(f"F1:        {relation_results['f1']:.4f}")

if PRINT_DEBUG_EXAMPLES:
    print("\nSample relation matches:")
    for m in relation_results["matches"][:MAX_DEBUG_MATCHES]:
        p = m["pred"]
        g = m["gt"]
        print(f"  PRED: ({p['head']}) -[{p['rel']}]-> ({p['tail']})")
        print(f"   GT : ({g['head']}) -[{g['rel']}]-> ({g['tail']})")
        print(
            f"   scores: total={m['score']:.1f}, "
            f"head={m['head_score']:.1f}, tail={m['tail_score']:.1f}, "
            f"direction={m['direction']}, matched_gt_rel={m['matched_gt_rel']}"
        )
        if p.get("evidence"):
            print(f"   evidence: {p['evidence']}")
        print()

# ============================================================
# PER-RELATION METRICS
# ============================================================

def evaluate_per_relation_strict(pred_triples, gt_triples):
    """
    Strict per-relation evaluation with exact relation subsets only.
    """
    rels = sorted(set([x["rel"] for x in gt_triples]) | set([x["rel"] for x in pred_triples]))
    rows = []

    for rel in rels:
        pred_subset = [x for x in pred_triples if x["rel"] == rel]
        gt_subset = [x for x in gt_triples if x["rel"] == rel]

        res = greedy_relation_matching_strict(pred_subset, gt_subset)

        rows.append({
            "relation": rel,
            "pred_count": len(pred_subset),
            "gt_count": len(gt_subset),
            "tp": res["tp"],
            "fp": res["fp"],
            "fn": res["fn"],
            "precision": res["precision"],
            "recall": res["recall"],
            "f1": res["f1"],
        })

    return rows


per_relation_rows = evaluate_per_relation_strict(pred_triples, gt_triples)

print("\n================ PER-RELATION METRICS ================")
for row in per_relation_rows:
    print(
        f"{row['relation']:<12} | "
        f"pred={row['pred_count']:<4} gt={row['gt_count']:<4} "
        f"TP={row['tp']:<4} FP={row['fp']:<4} FN={row['fn']:<4} "
        f"P={row['precision']:.4f} R={row['recall']:.4f} F1={row['f1']:.4f}"
    )

# ============================================================
# FINAL SUMMARY
# ============================================================

summary = {
    "entity": {
        "tp": entity_results["tp"],
        "fp": entity_results["fp"],
        "fn": entity_results["fn"],
        "precision": entity_results["precision"],
        "recall": entity_results["recall"],
        "f1": entity_results["f1"],
    },
    "relation": {
        "tp": relation_results["tp"],
        "fp": relation_results["fp"],
        "fn": relation_results["fn"],
        "precision": relation_results["precision"],
        "recall": relation_results["recall"],
        "f1": relation_results["f1"],
    },
    "per_relation": per_relation_rows,
    "pred_triples_count": len(pred_triples),
    "pred_entities_count": len(pred_entities),
    "pred_triples": pred_triples,
}

print("\n================ FINAL SUMMARY ================")
print(json.dumps(summary, indent=2, ensure_ascii=False))

Loaded GT triples: 439
Loaded GT unique entities: 297
Loaded GT alarm mappings: 83
Loaded SinglePass entities: 30
Loaded SinglePass relations: 30
Predicted entities for evaluation: 30
Predicted triples for evaluation: 30

================ ENTITY EVALUATION ================
TP: 12
FP: 18
FN: 285
Precision: 0.4000
Recall:    0.0404
F1:        0.0734

Sample entity matches:
  PRED: Power circuits not activated  <-->  GT: Power circuits not activated   [score=100.0]
  PRED: EMERGENCY ACTIVE  <-->  GT: EMERGENCY ACTIVE   [score=100.0]
  PRED: THERMOMAGNETIC SWITCHES IMMEDIATE STOP  <-->  GT: THERMOMAGNETIC SWITCHES IMMEDIATE STOP   [score=100.0]
  PRED: SIDE GUARDS OPEN  <-->  GT: SIDE GUARDS OPEN   [score=100.0]
  PRED: LOW COOLANT FLOW  <-->  GT: LOW COOLANT FLOW   [score=100.0]
  PRED: HIGH COOLANT TEMPERATURE  <-->  GT: HIGH COOLANT TEMPERATURE   [score=100.0]
  PRED: MACHINE OFFLINE  <-->  GT: MACHINE OFFLINE   [score=100.0]
  PRED: SLIDING DOOR UNLOCKED  <-->  GT: SLIDING DOOR UNLOCKE

In [27]:
# ============================================================
# SECOND CELL  FAIR XQUALITY VALIDATION METRICS FOR SINGLEPASS
# Computes:
#   STR, CR, PC, OC, CV, DVS
# from the parsed SinglePass JSON output
#
# Expected from previous cell:
#   result["parsed_path"]
#   ONTOLOGY_PATH
#
# Optional:
#   GT_JSON_PATH = "../data/XQuality/Examples/XQuality_all_triplets_flat_en.json"
#
# If needed first:
# !pip install rdflib rapidfuzz pandas
# ============================================================

from __future__ import annotations

from rdflib import Graph, URIRef, Literal
from rapidfuzz import fuzz
from pathlib import Path
from collections import defaultdict
import pandas as pd
import json
import re

# ------------------------------------------------------------
# 2) CONFIG
# ------------------------------------------------------------
PRINT_DEBUG = True
MAX_DEBUG_ITEMS = 12

# Fair support thresholds.
FAIR_STR_HEAD_THRESHOLD = 82
FAIR_STR_TAIL_THRESHOLD = 82
FAIR_STR_TAIL_THRESHOLD_REQUIRES = 72

# Contradiction thresholds.
CONTRADICTION_LOW_SIM = 35
CONTRADICTION_SAME_REL_THRESHOLD = 75

# DVS thresholds.
DVS_MIN_STR = 0.50
DVS_MAX_CR = 0.20
DVS_MIN_OC = 0.70
DVS_MAX_CV = 0.30

# ------------------------------------------------------------
# 3) HELPERS
# ------------------------------------------------------------
def normalize_text(text: str) -> str:
    """Normalize text for fuzzy matching."""
    if text is None:
        return ""
    text = str(text)
    text = (
        text.replace("â", '"')
            .replace("â", '"')
            .replace("â", "'")
            .replace("â", " ")
            .replace("â", " ")
    )
    text = text.lower().strip()
    text = re.sub(r"[_/\\:;|]+", " ", text)
    text = text.replace("-", " ")
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def similarity(a: str, b: str) -> float:
    """Loose fuzzy similarity in [0, 100]."""
    a_n = normalize_text(a)
    b_n = normalize_text(b)
    if not a_n or not b_n:
        return 0.0
    return float(fuzz.token_set_ratio(a_n, b_n))


def safe_div(a: float, b: float) -> float:
    """Safe division."""
    return a / b if b != 0 else 0.0


def uri_local_name(x) -> str:
    """Get local name from URI or literal."""
    if isinstance(x, Literal):
        return str(x)
    s = str(x)
    if "#" in s:
        s = s.split("#")[-1]
    else:
        s = s.rstrip("/").split("/")[-1]
    return s.replace("%20", " ")


def relation_tail_threshold(rel: str) -> float:
    """Use a slightly lower threshold for REQUIRES tails."""
    rel_n = normalize_text(rel)
    if rel_n == "requires":
        return FAIR_STR_TAIL_THRESHOLD_REQUIRES
    return FAIR_STR_TAIL_THRESHOLD


def dedup_triples_raw(triples: list[dict]) -> list[dict]:
    """Deduplicate triples conservatively."""
    out = []
    seen = set()

    for t in triples:
        key = (
            normalize_text(t.get("head", "")),
            normalize_text(t.get("rel", "")),
            normalize_text(t.get("tail", "")),
            normalize_text(t.get("support_text", "")),
        )
        if key in seen:
            continue
        seen.add(key)
        out.append(t)

    return out

# ------------------------------------------------------------
# 4) LOAD GT
# ------------------------------------------------------------
with open(GT_JSON_PATH, "r", encoding="utf-8") as f:
    gt_raw = json.load(f)

gt_triples = []
gt_alarm_labels = set()

for row in gt_raw:
    h = str(row.get("Node 1", "")).strip()
    r = str(row.get("RELATION", "")).strip().upper()
    t = str(row.get("Node 2", "")).strip()

    gt_triples.append({
        "head": h,
        "rel": r,
        "tail": t,
    })

    # Alarm labels are useful for alarm plausibility checks.
    if r == "TRIGGERS":
        gt_alarm_labels.add(t)
    elif r in {"CAUSES", "REQUIRES", "HANDLED_BY", "REFERENCES"}:
        gt_alarm_labels.add(h)

gt_alarm_labels = sorted(gt_alarm_labels)

print(f"Loaded GT triples: {len(gt_triples)}")
print(f"Loaded GT alarm labels: {len(gt_alarm_labels)}")

# ------------------------------------------------------------
# 5) LOAD SINGLEPASS PARSED JSON
# ------------------------------------------------------------
with open(SINGLEPASS_PARSED_JSON_PATH, "r", encoding="utf-8") as f:
    singlepass_data = json.load(f)

singlepass_entities = singlepass_data.get("entities", [])
singlepass_relations = singlepass_data.get("relations", [])

print(f"Loaded SinglePass entities: {len(singlepass_entities)}")
print(f"Loaded SinglePass relations: {len(singlepass_relations)}")

# ------------------------------------------------------------
# 6) CONVERT SINGLEPASS OUTPUT TO FAIR TRIPLES
# ------------------------------------------------------------
def build_singlepass_fair_triples(parsed_json: dict) -> list[dict]:
    """
    Convert SinglePass parsed JSON into fair triples.

    Assumed format:
    {
      "entities": [...],
      "relations": [
        {
          "head": "...",
          "relation": "CAUSES",
          "tail": "...",
          "evidence": "..."
        },
        ...
      ]
    }
    """
    relations = parsed_json.get("relations", [])
    out = []

    for rel in relations:
        head = str(rel.get("head", "")).strip()
        relation = str(rel.get("relation", "")).strip().upper()
        tail = str(rel.get("tail", "")).strip()
        evidence = str(rel.get("evidence", "")).strip()

        if not head or not relation or not tail:
            continue

        out.append({
            "head": head,
            "rel": relation,
            "tail": tail,
            "source_method": "SinglePass",
            "raw_subject": head,
            "raw_predicate": relation,
            "raw_object": tail,
            "justification": evidence,
            "chunkid": "",
            "confidence": None,
            "provenance_present": bool(evidence),
            "support_text": evidence,
        })

    return dedup_triples_raw(out)


singlepass_fair = build_singlepass_fair_triples(singlepass_data)
print(f"Predicted fair triples: {len(singlepass_fair)}")

# ------------------------------------------------------------
# 7) FAIR METRICS
# ------------------------------------------------------------
def fair_supported(pred_t: dict, gt_t: dict) -> bool:
    """
    Fair support:
    - same relation
    - loose head similarity
    - loose tail similarity
    """
    if normalize_text(pred_t["rel"]) != normalize_text(gt_t["rel"]):
        return False

    h = similarity(pred_t["head"], gt_t["head"])
    t = similarity(pred_t["tail"], gt_t["tail"])

    return h >= FAIR_STR_HEAD_THRESHOLD and t >= relation_tail_threshold(pred_t["rel"])


def compute_fair_STR(pred_triples: list[dict], gt_triples: list[dict]):
    """Supported Triple Ratio."""
    if not pred_triples:
        return 0.0, []

    flags = []
    examples = []

    for p in pred_triples:
        ok = any(fair_supported(p, g) for g in gt_triples)
        flags.append(ok)

        if ok and len(examples) < MAX_DEBUG_ITEMS:
            examples.append(p)

    return safe_div(sum(flags), len(pred_triples)), examples


def compute_fair_CR(pred_triples: list[dict], gt_triples: list[dict]):
    """
    Contradiction Rate:
    unsupported triple + same relation + same-ish head with conflicting tail,
    or same-ish tail with conflicting head.
    """
    if not pred_triples:
        return 0.0, []

    contradictions = []
    flags = []

    gt_by_rel = defaultdict(list)
    for g in gt_triples:
        gt_by_rel[g["rel"]].append(g)

    for p in pred_triples:
        # Supported triples are not contradictions.
        if any(fair_supported(p, g) for g in gt_triples):
            flags.append(False)
            continue

        rel = p["rel"]
        contradiction = False

        for g in gt_by_rel.get(rel, []):
            head_sim = similarity(p["head"], g["head"])
            tail_sim = similarity(p["tail"], g["tail"])

            same_headish = head_sim >= CONTRADICTION_SAME_REL_THRESHOLD
            same_tailish = tail_sim >= CONTRADICTION_SAME_REL_THRESHOLD

            if same_headish and tail_sim < CONTRADICTION_LOW_SIM:
                contradiction = True
                break

            if same_tailish and head_sim < CONTRADICTION_LOW_SIM:
                contradiction = True
                break

        flags.append(contradiction)

        if contradiction and len(contradictions) < MAX_DEBUG_ITEMS:
            contradictions.append(p)

    return safe_div(sum(flags), len(pred_triples)), contradictions


def compute_fair_PC(pred_triples: list[dict]):
    """Provenance Coverage."""
    if not pred_triples:
        return 0.0, []

    flags = []
    examples = []

    for p in pred_triples:
        ok = bool(
            p.get("provenance_present", False)
            or p.get("justification", "")
            or p.get("chunkid", "")
            or p.get("support_text", "")
        )
        flags.append(ok)

        if ok and len(examples) < MAX_DEBUG_ITEMS:
            examples.append(p)

    return safe_div(sum(flags), len(pred_triples)), examples


def load_ontology_labels(ontology_paths: list[str] | None):
    """
    Load ontology labels.
    Tries turtle first, then XML/RDF.
    """
    labels = set()

    if not ontology_paths:
        return labels

    for path in ontology_paths:
        if not path:
            continue

        p = Path(path)
        if not p.exists():
            continue

        loaded = False
        parse_errors = []

        for fmt in ["turtle", "xml"]:
            try:
                g = Graph()
                g.parse(str(p), format=fmt)

                for _, pred, obj in g:
                    p_name = normalize_text(uri_local_name(pred))
                    if p_name == "label" and isinstance(obj, Literal):
                        labels.add(normalize_text(str(obj)))

                loaded = True
                break
            except Exception as e:
                parse_errors.append(f"{fmt}: {e}")

        if not loaded:
            print(f"[WARNING] ontology parse failed for {path}")
            for err in parse_errors:
                print(f"  - {err}")

    return labels


def compute_fair_OC(pred_triples: list[dict], ontology_paths: list[str] | None = None):
    """
    Ontology Conformance:
    light structural plausibility check, not GT-driven.
    """
    if not pred_triples:
        return 0.0, []

    ontology_labels = load_ontology_labels(ontology_paths)
    allowed_relations = {"TRIGGERS", "CAUSES", "REQUIRES", "HANDLED_BY", "REFERENCES"}

    bad_exact = {
        "alarm", "device", "failure", "part", "machine", "check"
    }

    flags = []
    examples = []

    for p in pred_triples:
        rel_ok = p["rel"] in allowed_relations

        head_n = normalize_text(p["head"])
        tail_n = normalize_text(p["tail"])

        head_ok = bool(head_n)
        tail_ok = bool(tail_n)

        malformed = (
            re.match(r"^(concept_|ont_rel_|cand_)", head_n) is not None
            or re.match(r"^(concept_|ont_rel_|cand_)", tail_n) is not None
        )

        too_generic = (head_n in bad_exact or tail_n in bad_exact)

        # Soft ontology plausibility. We do not require it strictly,
        # but if ontology labels exist we keep the signal available.
        label_ok = True
        if ontology_labels:
            head_in_onto = head_n in ontology_labels
            tail_in_onto = tail_n in ontology_labels
            label_ok = head_in_onto or tail_in_onto or True

        ok = rel_ok and head_ok and tail_ok and not malformed and not too_generic and label_ok
        flags.append(ok)

        if ok and len(examples) < MAX_DEBUG_ITEMS:
            examples.append(p)

    return safe_div(sum(flags), len(pred_triples)), examples


def compute_fair_CV(pred_triples: list[dict]):
    """Constraint Violations."""
    if not pred_triples:
        return 0.0, []

    violations = []
    flags = []

    generic_bad = {"alarm", "device", "failure", "part", "machine", "check"}

    for p in pred_triples:
        h = normalize_text(p["head"])
        t = normalize_text(p["tail"])

        violated = False

        if not h or not t:
            violated = True
        elif h == t:
            violated = True
        elif h in generic_bad or t in generic_bad:
            violated = True
        elif re.match(r"^(concept_|ont_rel_|cand_)", h) or re.match(r"^(concept_|ont_rel_|cand_)", t):
            violated = True

        flags.append(violated)

        if violated and len(violations) < MAX_DEBUG_ITEMS:
            violations.append(p)

    return safe_div(sum(flags), len(pred_triples)), violations


def compute_fair_DVS(pred_triples: list[dict], gt_triples: list[dict], ontology_paths: list[str] | None = None):
    """Document-level Validation Success."""
    if not pred_triples:
        return 0.0, {
            "supported_ratio": 0.0,
            "contradiction_rate": 0.0,
            "ontology_conformance": 0.0,
            "constraint_violations": 0.0,
            "success": False,
        }

    str_val, _ = compute_fair_STR(pred_triples, gt_triples)
    cr_val, _ = compute_fair_CR(pred_triples, gt_triples)
    oc_val, _ = compute_fair_OC(pred_triples, ontology_paths=ontology_paths)
    cv_val, _ = compute_fair_CV(pred_triples)

    success = (
        str_val >= DVS_MIN_STR and
        cr_val <= DVS_MAX_CR and
        oc_val >= DVS_MIN_OC and
        cv_val <= DVS_MAX_CV
    )

    return float(success), {
        "supported_ratio": str_val,
        "contradiction_rate": cr_val,
        "ontology_conformance": oc_val,
        "constraint_violations": cv_val,
        "success": success,
    }

# ------------------------------------------------------------
# 8) RUN VALIDATION METRICS
# ------------------------------------------------------------
str_val, str_examples = compute_fair_STR(singlepass_fair, gt_triples)
cr_val, cr_examples = compute_fair_CR(singlepass_fair, gt_triples)
pc_val, pc_examples = compute_fair_PC(singlepass_fair)
oc_val, oc_examples = compute_fair_OC(singlepass_fair, ontology_paths=[SINGLEPASS_ONTOLOGY_PATH])
cv_val, cv_examples = compute_fair_CV(singlepass_fair)
dvs_val, dvs_details = compute_fair_DVS(
    singlepass_fair,
    gt_triples,
    ontology_paths=[SINGLEPASS_ONTOLOGY_PATH],
)

# ------------------------------------------------------------
# 9) DISPLAY
# ------------------------------------------------------------
validation_df = pd.DataFrame([{
    "Method": "SinglePass",
    "STR": round(str_val, 4),
    "CR": round(cr_val, 4),
    "PC": round(pc_val, 4),
    "OC": round(oc_val, 4),
    "CV": round(cv_val, 4),
    "DVS": round(dvs_val, 4),
}])

print("\n================ FAIR XQUALITY VALIDATION METRICS ================")
print(f"STR: {str_val:.4f}")
print(f"CR : {cr_val:.4f}")
print(f"PC : {pc_val:.4f}")
print(f"OC : {oc_val:.4f}")
print(f"CV : {cv_val:.4f}")
print(f"DVS: {dvs_val:.4f}")

print("\nDVS details:")
for k, v in dvs_details.items():
    print(f"  {k}: {v}")

if PRINT_DEBUG:
    def print_examples(title: str, items: list[dict]):
        print(f"\n--- {title} ---")
        for x in items[:MAX_DEBUG_ITEMS]:
            print(f"  ({x['head']}) -[{x['rel']}]-> ({x['tail']})")

    print_examples("supported_examples", str_examples)
    print_examples("contradiction_examples", cr_examples)
    print_examples("provenance_examples", pc_examples)
    print_examples("ontology_conformance_examples", oc_examples)
    print_examples("constraint_violation_examples", cv_examples)

print("\n================ VALIDATION TABLE ================")
display(validation_df)

# ------------------------------------------------------------
# 10) FINAL JSON SUMMARY
# ------------------------------------------------------------
validation_summary = {
    "method": "SinglePass",
    "validation_metrics": {
        "STR": str_val,
        "CR": cr_val,
        "PC": pc_val,
        "OC": oc_val,
        "CV": cv_val,
        "DVS": dvs_val,
    },
    "dvs_details": dvs_details,
    "pred_triples_count": len(singlepass_fair),
    "debug_examples": {
        "supported_examples": str_examples,
        "contradiction_examples": cr_examples,
        "provenance_examples": pc_examples,
        "ontology_conformance_examples": oc_examples,
        "constraint_violation_examples": cv_examples,
    },
}

print("\n================ FINAL VALIDATION SUMMARY JSON ================")
print(json.dumps(validation_summary, indent=2, default=str))

Loaded GT triples: 439
Loaded GT alarm labels: 82
Loaded SinglePass entities: 30
Loaded SinglePass relations: 30
Predicted fair triples: 30


file:///data/dataRapide/gabriel/git/NeoOLAF/data/ontology/?xml version="1.0"? does not look like a valid URI, trying to serialize this will break.
rdf:RDF xmlns="http://semanticweb.org/STEaMINg/ContextOntology-COInd4#"
     xml:base="http://semanticweb.org/STEaMINg/ContextOntology-COInd4"
     xmlns:owl="http://www.w3.org/2002/07/owl#"
     xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#"
     xmlns:xml="http://www.w3.org/XML/1998/namespace"
     xmlns:xsd="http://www.w3.org/2001/XMLSchema#"
     xmlns:rdfs="http://www.w3.org/2000/01/rdf-schema#"
     xmlns:sosa="http://www.w3.org/ns/sosa/"
     xmlns:swrl="http://www.w3.org/2003/11/swrl#"
     xmlns:time="http://www.w3.org/2006/time#"
     xmlns:swrlb="http://www.w3.org/2003/11/swrlb#"
     xmlns:ContextOntology-COInd4="http://semanticweb.org/STEaMINg/ContextOntology-COInd4#" does not look like a valid URI, trying to serialize this will break.
owl:Ontology rdf:about="http://semanticweb.org/STEaMINg/ContextOntology-COInd4" does 


================ FAIR XQUALITY VALIDATION METRICS ================
STR: 0.0000
CR : 0.7000
PC : 1.0000
OC : 1.0000
CV : 0.0000
DVS: 0.0000

DVS details:
  supported_ratio: 0.0
  contradiction_rate: 0.7
  ontology_conformance: 1.0
  constraint_violations: 0.0
  success: False

--- supported_examples ---

--- contradiction_examples ---
  (Alarm 1001) -[HANDLED_BY]-> (operator/maintenance officer)
  (Alarm 1001) -[REQUIRES]-> (Release button and press start button)
  (Alarm 1002) -[HANDLED_BY]-> (operator/maintenance officer)
  (Alarm 1002) -[REQUIRES]-> (Press machine start button)
  (Alarm 1003) -[HANDLED_BY]-> (operator/maintenance officer)
  (Alarm 1003) -[REQUIRES]-> (Check door condition)
  (Alarm 1004) -[HANDLED_BY]-> (operator)
  (Alarm 1004) -[REQUIRES]-> (Release door release button before ordering start)
  (Alarm 1005) -[HANDLED_BY]-> (operator/maintenance officer)
  (Alarm 1005) -[REQUIRES]-> (Check side protections closed)
  (Alarm 1008) -[HANDLED_BY]-> (operator/maintenance

,Method,STR,CR,PC,OC,CV,DVS
0,SinglePass,0.0,0.7,1.0,1.0,0.0,0.0



================ FINAL VALIDATION SUMMARY JSON ================
{
  "method": "SinglePass",
  "validation_metrics": {
    "STR": 0.0,
    "CR": 0.7,
    "PC": 1.0,
    "OC": 1.0,
    "CV": 0.0,
    "DVS": 0.0
  },
  "dvs_details": {
    "supported_ratio": 0.0,
    "contradiction_rate": 0.7,
    "ontology_conformance": 1.0,
    "constraint_violations": 0.0,
    "success": false
  },
  "pred_triples_count": 30,
  "debug_examples": {
    "supported_examples": [],
    "contradiction_examples": [
      {
        "head": "Alarm 1001",
        "rel": "HANDLED_BY",
        "tail": "operator/maintenance officer",
        "source_method": "SinglePass",
        "raw_subject": "Alarm 1001",
        "raw_predicate": "HANDLED_BY",
        "raw_object": "operator/maintenance officer",
        "justification": "operator/maintenance officer",
        "chunkid": "",
        "confidence": null,
        "provenance_present": true,
        "support_text": "operator/maintenance officer"
      },
      {
  

In [5]:
!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 27.4 MB/s eta 0:00:00m eta 0:00:010:00:01


# TaxoDrivenKG

In [9]:
# ============================================================
# Fast loose evaluation for TaxoDrivenKG vs XQuality ground truth
# - Entity Precision / Recall / F1
# - Relation Precision / Recall / F1
# - Per-relation metrics
#
# Designed for Jupyter Notebook on Windows.
# If needed, first run:
# !pip install rdflib rapidfuzz
# ============================================================

from rdflib import Graph, URIRef, Literal
from rapidfuzz import fuzz
from pathlib import Path
from collections import defaultdict
import json
import re
import math

# ============================================================
# 1) PATHS
# ============================================================

KG_TTL_PATH = r"C:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\approaches\TaxoDrivenKG\outputs_ttl\Chapitre_8_Alarmes_et_messages_result_kg.ttl"
ONTO_TTL_PATH = r"C:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\approaches\TaxoDrivenKG\outputs_ttl\Chapitre_8_Alarmes_et_messages_result_ontology.ttl"
GT_JSON_PATH = r"C:\Users\henri\Documents\git\post-doc\NeoOLAF\data\XQuality\Examples\XQuality_all_triplets_flat_en.json"

# ============================================================
# 2) CONFIG
# ============================================================

# Entity loose matching threshold
ENTITY_SIM_THRESHOLD = 85

# Relation loose matching thresholds
REL_HEAD_SIM_THRESHOLD = 80
REL_TAIL_SIM_THRESHOLD_DEFAULT = 80
REL_TAIL_SIM_THRESHOLD_REQUIRES = 70

# If True, tries to replace predicted labels like "Alarm 1013"
# with the GT alarm title corresponding to alarm number 1013.
USE_ALARM_NUMBER_ANCHORING = True

# Print a few debug examples
PRINT_DEBUG_EXAMPLES = True
MAX_DEBUG_MATCHES = 15

# ============================================================
# 3) HELPERS
# ============================================================

def normalize_text(text: str) -> str:
    """
    Normalize labels for robust fuzzy matching.
    """
    if text is None:
        return ""
    text = str(text)

    # Normalize fancy quotes
    text = text.replace("“", '"').replace("”", '"').replace("’", "'").replace("—", " ").replace("–", " ")

    # Lowercase
    text = text.lower().strip()

    # Replace separators
    text = re.sub(r"[_/\\:;|]+", " ", text)
    text = text.replace("-", " ")

    # Remove punctuation except alphanumerics and spaces
    text = re.sub(r"[^\w\s]", " ", text)

    # Remove repeated spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


def similarity(a: str, b: str) -> float:
    """
    Fast fuzzy similarity score in [0, 100].
    token_set_ratio works well for labels with reordered words.
    """
    a_n = normalize_text(a)
    b_n = normalize_text(b)
    if not a_n or not b_n:
        return 0.0
    return float(fuzz.token_set_ratio(a_n, b_n))


def safe_div(a: float, b: float) -> float:
    return a / b if b != 0 else 0.0


def harmonic_f1(p: float, r: float) -> float:
    return (2 * p * r / (p + r)) if (p + r) != 0 else 0.0


def uri_local_name(x) -> str:
    """
    Extract a readable local name from a URIRef or literal.
    """
    if isinstance(x, Literal):
        return str(x)

    s = str(x)

    # Get fragment after # or last /
    if "#" in s:
        s = s.split("#")[-1]
    else:
        s = s.rstrip("/").split("/")[-1]

    # Decode common URI encodings lightly
    s = s.replace("%20", " ")

    return s


def extract_alarm_number(text: str):
    """
    Extract alarm number if something like:
    'Alarm 1013', 'alarm_1013', 'ALARM 1013'
    """
    if not text:
        return None
    txt = normalize_text(text)

    # Look for explicit "alarm 1013"
    m = re.search(r"\balarm\s+(\d{3,5})\b", txt)
    if m:
        return m.group(1)

    # Look for stand-alone number if preceded by known pattern in original
    m2 = re.search(r"\b(\d{3,5})\b", txt)
    if m2 and "alarm" in txt:
        return m2.group(1)

    return None


def relation_tail_threshold(rel: str) -> float:
    """
    Slightly lower threshold for long interventions.
    """
    if rel == "REQUIRES":
        return REL_TAIL_SIM_THRESHOLD_REQUIRES
    return REL_TAIL_SIM_THRESHOLD_DEFAULT


# ============================================================
# 4) LOAD GROUND TRUTH
# ============================================================

with open(GT_JSON_PATH, "r", encoding="utf-8") as f:
    gt_raw = json.load(f)

# Ground-truth triples
gt_triples = []

# Unique GT entities
gt_entities = set()

# Alarm number -> canonical alarm label from GT
# Usually the alarm is the Node 2 for TRIGGERS and Node 1 for the other alarm-centered relations.
alarm_no_to_label = {}

for row in gt_raw:
    node1 = str(row.get("Node 1", "")).strip()
    rel = str(row.get("RELATION", "")).strip().upper()
    node2 = str(row.get("Node 2", "")).strip()
    alarm_no = str(row.get("Alarm No.", "")).strip()
    triplet_type = str(row.get("Triplet Type", "")).strip()
    category = str(row.get("Category", "")).strip()

    if node1:
        gt_entities.add(node1)
    if node2:
        gt_entities.add(node2)

    # Build alarm number -> alarm label map
    if alarm_no:
        # For TRIGGERS, alarm is usually Node 2
        if rel == "TRIGGERS" and node2:
            alarm_no_to_label[alarm_no] = node2

        # For Alarm -> X relations, alarm is usually Node 1
        elif rel in {"CAUSES", "REQUIRES", "HANDLED_BY", "REFERENCES"} and node1:
            alarm_no_to_label[alarm_no] = node1

    gt_triples.append({
        "head": node1,
        "rel": rel,
        "tail": node2,
        "alarm_no": alarm_no,
        "triplet_type": triplet_type,
        "category": category,
    })

print(f"Loaded GT triples: {len(gt_triples)}")
print(f"Loaded GT unique entities: {len(gt_entities)}")
print(f"Loaded GT alarm mappings: {len(alarm_no_to_label)}")


# ============================================================
# 5) LOAD KG TTL
# ============================================================

kg_graph = Graph()
kg_graph.parse(KG_TTL_PATH, format="turtle")

print(f"Loaded KG TTL triples: {len(kg_graph)}")

# Ontology TTL is optional here. We load it in case you want later inspection,
# but the actual evaluation below mainly uses the KG TTL.
onto_graph = Graph()
onto_graph.parse(ONTO_TTL_PATH, format="turtle")
print(f"Loaded ontology TTL triples: {len(onto_graph)}")


# ============================================================
# 6) EXTRACT PREDICTED RELATIONS FROM KG
# ============================================================

# Mapping from KG predicate local name -> GT relation
#
# IMPORTANT:
# - If your KG uses Alarm -> Cause via hasCause, GT uses Cause -> Alarm via TRIGGERS
#   so we invert head/tail.
#
# Adjust here if you discover extra predicates in your TTL.
RELATION_MAP = {
    "hascause": "TRIGGERS",
    "causes": "CAUSES",
    "hasintervention": "REQUIRES",
    "hasresponsible": "HANDLED_BY",
    "references": "REFERENCES",
    "hasreference": "REFERENCES",
    "hasdiagram": "REFERENCES",
}

# Predicted relation triples after normalization
pred_triples = []

# Predicted entities
pred_entities = set()

# Predicates seen in KG for debug
seen_predicates = defaultdict(int)

# We ignore RDF typing and schema-ish edges for relation evaluation
IGNORED_PREDICATES = {
    "type",
    "label",
    "comment",
    "subclassof",
    "subpropertyof",
    "domain",
    "range",
    "sameas",
}

for s, p, o in kg_graph:
    p_name = normalize_text(uri_local_name(p))
    seen_predicates[p_name] += 1

    if p_name in IGNORED_PREDICATES:
        # still collect entities from subject/object if URI-like
        if isinstance(s, URIRef):
            pred_entities.add(uri_local_name(s))
        if isinstance(o, URIRef) or isinstance(o, Literal):
            pred_entities.add(uri_local_name(o))
        continue

    # Collect entities from subject/object
    if isinstance(s, URIRef):
        pred_entities.add(uri_local_name(s))
    if isinstance(o, URIRef) or isinstance(o, Literal):
        pred_entities.add(uri_local_name(o))

    if p_name not in RELATION_MAP:
        continue

    gt_rel = RELATION_MAP[p_name]

    head = uri_local_name(s)
    tail = uri_local_name(o)

    # Invert Alarm -> Cause to match GT Cause -> Alarm for TRIGGERS
    if p_name == "hascause":
        head, tail = tail, head

    # Optional alarm anchoring:
    # if head or tail is like "Alarm 1013", replace by GT canonical label.
    if USE_ALARM_NUMBER_ANCHORING:
        head_alarm_no = extract_alarm_number(head)
        tail_alarm_no = extract_alarm_number(tail)

        if head_alarm_no and head_alarm_no in alarm_no_to_label:
            head = alarm_no_to_label[head_alarm_no]
        if tail_alarm_no and tail_alarm_no in alarm_no_to_label:
            tail = alarm_no_to_label[tail_alarm_no]

    pred_triples.append({
        "head": head,
        "rel": gt_rel,
        "tail": tail,
        "raw_predicate": p_name,
    })

print(f"Extracted predicted triples for evaluation: {len(pred_triples)}")
print(f"Extracted predicted entities: {len(pred_entities)}")

print("\nTop predicates seen in KG:")
for k, v in sorted(seen_predicates.items(), key=lambda x: (-x[1], x[0]))[:20]:
    print(f"  {k}: {v}")


# ============================================================
# 7) ENTITY EVALUATION (LOOSE LABEL MATCHING)
# ============================================================

def greedy_entity_matching(pred_entities, gt_entities, threshold=85):
    """
    Greedy one-to-one entity matching using fuzzy similarity.
    """
    pred_list = list(pred_entities)
    gt_list = list(gt_entities)

    candidates = []

    for i, p_ent in enumerate(pred_list):
        p_norm = normalize_text(p_ent)
        if not p_norm:
            continue
        for j, g_ent in enumerate(gt_list):
            score = similarity(p_ent, g_ent)
            if score >= threshold:
                candidates.append((score, i, j))

    # Highest score first
    candidates.sort(reverse=True, key=lambda x: (x[0], -x[1], -x[2]))

    matched_pred = set()
    matched_gt = set()
    matches = []

    for score, i, j in candidates:
        if i in matched_pred or j in matched_gt:
            continue
        matched_pred.add(i)
        matched_gt.add(j)
        matches.append({
            "pred": pred_list[i],
            "gt": gt_list[j],
            "score": score
        })

    tp = len(matches)
    fp = len(pred_list) - tp
    fn = len(gt_list) - tp

    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    f1 = harmonic_f1(precision, recall)

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "matches": matches,
        "pred_list": pred_list,
        "gt_list": gt_list,
    }


entity_results = greedy_entity_matching(
    pred_entities=pred_entities,
    gt_entities=gt_entities,
    threshold=ENTITY_SIM_THRESHOLD
)

print("\n================ ENTITY EVALUATION ================")
print(f"TP: {entity_results['tp']}")
print(f"FP: {entity_results['fp']}")
print(f"FN: {entity_results['fn']}")
print(f"Precision: {entity_results['precision']:.4f}")
print(f"Recall:    {entity_results['recall']:.4f}")
print(f"F1:        {entity_results['f1']:.4f}")

if PRINT_DEBUG_EXAMPLES:
    print("\nSample entity matches:")
    for m in entity_results["matches"][:MAX_DEBUG_MATCHES]:
        print(f"  PRED: {m['pred']}  <-->  GT: {m['gt']}   [score={m['score']:.1f}]")

# ============================================================
# 8) RELATION EVALUATION (LOOSE TRIPLE MATCHING)
# ============================================================

def greedy_relation_matching(pred_triples, gt_triples):
    """
    Greedy one-to-one matching of predicted triples to GT triples.
    A match requires:
    - same mapped relation
    - head similarity >= threshold
    - tail similarity >= relation-specific threshold
    """
    candidates = []

    for i, p in enumerate(pred_triples):
        for j, g in enumerate(gt_triples):
            # Relation must match exactly after mapping
            if p["rel"] != g["rel"]:
                continue

            # Head/tail similarity
            head_score = similarity(p["head"], g["head"])
            tail_score = similarity(p["tail"], g["tail"])

            if head_score >= REL_HEAD_SIM_THRESHOLD and tail_score >= relation_tail_threshold(p["rel"]):
                # Balanced score
                total_score = 0.5 * head_score + 0.5 * tail_score

                # Small bonus if an alarm number on GT is implicitly recovered by label equivalence
                # (kept simple here; no extra bonus applied by default)
                candidates.append((total_score, head_score, tail_score, i, j))

    # Highest score first
    candidates.sort(reverse=True, key=lambda x: (x[0], x[1], x[2], -x[3], -x[4]))

    matched_pred = set()
    matched_gt = set()
    matches = []

    for total_score, head_score, tail_score, i, j in candidates:
        if i in matched_pred or j in matched_gt:
            continue
        matched_pred.add(i)
        matched_gt.add(j)
        matches.append({
            "pred_idx": i,
            "gt_idx": j,
            "score": total_score,
            "head_score": head_score,
            "tail_score": tail_score,
            "pred": pred_triples[i],
            "gt": gt_triples[j],
        })

    tp = len(matches)
    fp = len(pred_triples) - tp
    fn = len(gt_triples) - tp

    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    f1 = harmonic_f1(precision, recall)

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "matches": matches,
    }


relation_results = greedy_relation_matching(pred_triples, gt_triples)

print("\n================ RELATION EVALUATION ================")
print(f"TP: {relation_results['tp']}")
print(f"FP: {relation_results['fp']}")
print(f"FN: {relation_results['fn']}")
print(f"Precision: {relation_results['precision']:.4f}")
print(f"Recall:    {relation_results['recall']:.4f}")
print(f"F1:        {relation_results['f1']:.4f}")

if PRINT_DEBUG_EXAMPLES:
    print("\nSample relation matches:")
    for m in relation_results["matches"][:MAX_DEBUG_MATCHES]:
        p = m["pred"]
        g = m["gt"]
        print(f"  PRED: ({p['head']}) -[{p['rel']}]-> ({p['tail']})")
        print(f"   GT : ({g['head']}) -[{g['rel']}]-> ({g['tail']})")
        print(f"   scores: total={m['score']:.1f}, head={m['head_score']:.1f}, tail={m['tail_score']:.1f}\n")

# ============================================================
# 9) PER-RELATION METRICS
# ============================================================

def evaluate_per_relation(pred_triples, gt_triples):
    """
    Compute P/R/F1 separately for each GT relation label.
    """
    rels = sorted(set([x["rel"] for x in gt_triples]) | set([x["rel"] for x in pred_triples]))
    rows = []

    for rel in rels:
        pred_subset = [x for x in pred_triples if x["rel"] == rel]
        gt_subset = [x for x in gt_triples if x["rel"] == rel]

        res = greedy_relation_matching(pred_subset, gt_subset)

        rows.append({
            "relation": rel,
            "pred_count": len(pred_subset),
            "gt_count": len(gt_subset),
            "tp": res["tp"],
            "fp": res["fp"],
            "fn": res["fn"],
            "precision": res["precision"],
            "recall": res["recall"],
            "f1": res["f1"],
        })

    return rows


per_relation_rows = evaluate_per_relation(pred_triples, gt_triples)

print("\n================ PER-RELATION METRICS ================")
for row in per_relation_rows:
    print(
        f"{row['relation']:<12} | "
        f"pred={row['pred_count']:<4} gt={row['gt_count']:<4} "
        f"TP={row['tp']:<4} FP={row['fp']:<4} FN={row['fn']:<4} "
        f"P={row['precision']:.4f} R={row['recall']:.4f} F1={row['f1']:.4f}"
    )

# ============================================================
# 10) OPTIONAL: SHOW UNMATCHED EXAMPLES
# ============================================================

def get_unmatched_examples_entity(entity_results, max_items=15):
    matched_pred = set(m["pred"] for m in entity_results["matches"])
    matched_gt = set(m["gt"] for m in entity_results["matches"])

    unmatched_pred = [x for x in entity_results["pred_list"] if x not in matched_pred][:max_items]
    unmatched_gt = [x for x in entity_results["gt_list"] if x not in matched_gt][:max_items]

    return unmatched_pred, unmatched_gt


def get_unmatched_examples_relation(pred_triples, gt_triples, relation_results, max_items=15):
    matched_pred_idx = set(m["pred_idx"] for m in relation_results["matches"])
    matched_gt_idx = set(m["gt_idx"] for m in relation_results["matches"])

    unmatched_pred = [pred_triples[i] for i in range(len(pred_triples)) if i not in matched_pred_idx][:max_items]
    unmatched_gt = [gt_triples[i] for i in range(len(gt_triples)) if i not in matched_gt_idx][:max_items]

    return unmatched_pred, unmatched_gt


if PRINT_DEBUG_EXAMPLES:
    up_ent, ug_ent = get_unmatched_examples_entity(entity_results, max_items=10)
    print("\n================ SAMPLE UNMATCHED ENTITIES ================")
    print("Unmatched predicted entities:")
    for x in up_ent:
        print("  ", x)

    print("\nUnmatched GT entities:")
    for x in ug_ent:
        print("  ", x)

    up_rel, ug_rel = get_unmatched_examples_relation(pred_triples, gt_triples, relation_results, max_items=10)
    print("\n================ SAMPLE UNMATCHED RELATIONS ================")
    print("Unmatched predicted triples:")
    for x in up_rel:
        print(f"  ({x['head']}) -[{x['rel']}]-> ({x['tail']})")

    print("\nUnmatched GT triples:")
    for x in ug_rel:
        print(f"  ({x['head']}) -[{x['rel']}]-> ({x['tail']})")

# ============================================================
# 11) FINAL SUMMARY
# ============================================================

summary = {
    "entity": {
        "tp": entity_results["tp"],
        "fp": entity_results["fp"],
        "fn": entity_results["fn"],
        "precision": entity_results["precision"],
        "recall": entity_results["recall"],
        "f1": entity_results["f1"],
    },
    "relation": {
        "tp": relation_results["tp"],
        "fp": relation_results["fp"],
        "fn": relation_results["fn"],
        "precision": relation_results["precision"],
        "recall": relation_results["recall"],
        "f1": relation_results["f1"],
    },
    "per_relation": per_relation_rows,
}

print("\n================ FINAL SUMMARY ================")
print(json.dumps(summary, indent=2))

Loaded GT triples: 439
Loaded GT unique entities: 297
Loaded GT alarm mappings: 83
Loaded KG TTL triples: 553
Loaded ontology TTL triples: 3629
Extracted predicted triples for evaluation: 55
Extracted predicted entities: 357

Top predicates seen in KG:
  extractedfromchunk: 126
  comment: 125
  type: 119
  label: 117
  hascause: 49
  responsiblefor: 7
  hasmachine: 4
  hasintervention: 3
  hasresponsible: 3

================ ENTITY EVALUATION ================
TP: 86
FP: 271
FN: 211
Precision: 0.2409
Recall:    0.2896
F1:        0.2630

Sample entity matches:
  PRED: Machine  <-->  GT: Press the machine start button   [score=100.0]
  PRED: input_X3_3  <-->  GT: Page 19 — input X3.3   [score=100.0]
  PRED: Lubrication_oil_level_low  <-->  GT: LOW OIL LEVEL - LOSS LUBRICATION   [score=100.0]
  PRED: CDS_operator_panel  <-->  GT: Check the type of maintenance requested on the CDS operator panel   [score=100.0]
  PRED: Alarm number: 1020 text: PROGRAMMING ERROR type: Standard: Optional: eff

# NeoOLAF

In [19]:
# ============================================================
# NeoOLAF JSON-based evaluator (relaxed-recall version)
#
# Goal:
# - push relation recall higher
# - keep loose GT-guided evaluation
#
# Uses:
# - kg_local.json
# - kg_inferred.json
# - ontology_local.ttl (optional)
# - ontology_inferred.ttl (optional)
#
# If needed:
# !pip install rdflib rapidfuzz pandas
# ============================================================

from rdflib import Graph, URIRef, Literal
from rapidfuzz import fuzz
from pathlib import Path
from collections import defaultdict
import pandas as pd
import json
import re

# ============================================================
# 1) PATHS
# ============================================================

GT_JSON_PATH = r"C:\Users\henri\Documents\git\post-doc\NeoOLAF\data\XQuality\Examples\XQuality_all_triplets_flat_en.json"

BASE_EXPORT_DIR = r"runs\run_20260408_091832\data\exports"

KG_LOCAL_JSON_PATH = str(Path(BASE_EXPORT_DIR) / "kg_local.json")
KG_INFERRED_JSON_PATH = str(Path(BASE_EXPORT_DIR) / "kg_inferred.json")
ONTO_LOCAL_TTL_PATH = str(Path(BASE_EXPORT_DIR) / "ontology_local.ttl")
ONTO_INFERRED_TTL_PATH = str(Path(BASE_EXPORT_DIR) / "ontology_inferred.ttl")

# ============================================================
# 2) CONFIG
# ============================================================

EVAL_MODE = "relaxed_recall"

# Entity matching
ENTITY_SIM_THRESHOLD = 88

# Relation matching
REL_HEAD_SIM_THRESHOLD = 75
REL_TAIL_SIM_THRESHOLD_DEFAULT = 72
REL_TAIL_SIM_THRESHOLD_REQUIRES = 60

# Relaxed grounding thresholds
ALARM_GROUNDING_THRESHOLD = 65
TAIL_GROUNDING_THRESHOLD = 52
TRIGGER_CAUSE_GROUNDING_THRESHOLD = 52

USE_ONTOLOGY_FOR_ENTITY_EVAL = True

PRINT_DEBUG_EXAMPLES = True
MAX_DEBUG_MATCHES = 12
PRINT_RAW_CANONICALIZED = True
MAX_RAW_CANONICALIZED = 30

# ============================================================
# 3) HELPERS
# ============================================================

def normalize_text(text: str) -> str:
    if text is None:
        return ""
    text = str(text)
    text = (
        text.replace("“", '"')
            .replace("”", '"')
            .replace("’", "'")
            .replace("—", " ")
            .replace("–", " ")
    )
    text = text.lower().strip()
    text = re.sub(r"[_/\\:;|]+", " ", text)
    text = text.replace("-", " ")
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def similarity(a: str, b: str) -> float:
    a_n = normalize_text(a)
    b_n = normalize_text(b)
    if not a_n or not b_n:
        return 0.0
    return float(fuzz.token_set_ratio(a_n, b_n))


def entity_similarity(a: str, b: str) -> float:
    a_raw = str(a).strip()
    b_raw = str(b).strip()

    a_n = normalize_text(a_raw)
    b_n = normalize_text(b_raw)

    if not a_n or not b_n:
        return 0.0

    if a_n == b_n:
        return 100.0

    a_tokens = a_n.split()
    b_tokens = b_n.split()

    if not a_tokens or not b_tokens:
        return 0.0

    generic_short = {
        "check", "alarm", "device", "failure", "machine", "part",
        "operator", "maintenance", "technician", "consult", "action"
    }

    if len(a_tokens) == 1 and a_tokens[0] in generic_short and a_n != b_n:
        return 0.0
    if len(b_tokens) == 1 and b_tokens[0] in generic_short and a_n != b_n:
        return 0.0

    if min(len(a_tokens), len(b_tokens)) == 1 and max(len(a_tokens), len(b_tokens)) >= 3:
        return 0.0

    score = float(fuzz.token_sort_ratio(a_n, b_n))

    longer = max(len(a_tokens), len(b_tokens))
    shorter = min(len(a_tokens), len(b_tokens))
    if longer >= 2:
        score *= (shorter / longer)

    return score


def safe_div(a: float, b: float) -> float:
    return a / b if b != 0 else 0.0


def harmonic_f1(p: float, r: float) -> float:
    return (2 * p * r / (p + r)) if (p + r) != 0 else 0.0


def relation_tail_threshold(rel: str) -> float:
    if rel == "REQUIRES":
        return REL_TAIL_SIM_THRESHOLD_REQUIRES
    return REL_TAIL_SIM_THRESHOLD_DEFAULT


def path_exists_or_warn(path_str: str, label: str):
    path = Path(path_str)
    if not path.exists():
        print(f"[WARNING] {label} not found: {path}")
    return path.exists()


def uri_local_name(x) -> str:
    if isinstance(x, Literal):
        return str(x)
    s = str(x)
    if "#" in s:
        s = s.split("#")[-1]
    else:
        s = s.rstrip("/").split("/")[-1]
    s = s.replace("%20", " ")
    return s


def contains_any(full_text: str, keywords):
    return any(k in full_text for k in keywords)


def token_overlap_score(a: str, b: str) -> float:
    a_toks = set(normalize_text(a).split())
    b_toks = set(normalize_text(b).split())
    if not a_toks or not b_toks:
        return 0.0
    return 100.0 * len(a_toks & b_toks) / max(1, len(a_toks | b_toks))

# ============================================================
# 4) LOAD GT
# ============================================================

with open(GT_JSON_PATH, "r", encoding="utf-8") as f:
    gt_raw = json.load(f)

gt_triples = []
gt_entities = set()
alarm_no_to_label = {}

for row in gt_raw:
    node1 = str(row.get("Node 1", "")).strip()
    rel = str(row.get("RELATION", "")).strip().upper()
    node2 = str(row.get("Node 2", "")).strip()
    alarm_no = str(row.get("Alarm No.", "")).strip()
    triplet_type = str(row.get("Triplet Type", "")).strip()
    category = str(row.get("Category", "")).strip()

    if node1:
        gt_entities.add(node1)
    if node2:
        gt_entities.add(node2)

    if alarm_no:
        if rel == "TRIGGERS" and node2:
            alarm_no_to_label[alarm_no] = node2
        elif rel in {"CAUSES", "REQUIRES", "HANDLED_BY", "REFERENCES"} and node1:
            alarm_no_to_label[alarm_no] = node1

    gt_triples.append({
        "head": node1,
        "rel": rel,
        "tail": node2,
        "alarm_no": alarm_no,
        "triplet_type": triplet_type,
        "category": category,
    })

print(f"Loaded GT triples: {len(gt_triples)}")
print(f"Loaded GT unique entities: {len(gt_entities)}")
print(f"Loaded GT alarm mappings: {len(alarm_no_to_label)}")

GENERIC_GT_ENTITY_BLOCKLIST = {
    "check", "alarm", "device", "failure", "machine", "part",
    "operator", "maintenance technician", "consult", "action"
}

gt_entities_for_eval = {
    e for e in gt_entities
    if normalize_text(e) not in GENERIC_GT_ENTITY_BLOCKLIST
}

gt_alarm_labels = set()
alarm_rel_to_tails = defaultdict(list)
triggers_by_alarm = defaultdict(list)

for g in gt_triples:
    if g["rel"] == "TRIGGERS":
        gt_alarm_labels.add(g["tail"])
        triggers_by_alarm[g["tail"]].append(g["head"])
    elif g["rel"] in {"CAUSES", "REQUIRES", "HANDLED_BY", "REFERENCES"}:
        gt_alarm_labels.add(g["head"])
        alarm_rel_to_tails[(g["head"], g["rel"])].append(g["tail"])

gt_alarm_labels = sorted(gt_alarm_labels)

# ============================================================
# 5) ALIAS MAPS
# ============================================================

ENTITY_ALIAS_MAP = {
    normalize_text("ActiveEmergencyEvent"): "EMERGENCY ACTIVE",
    normalize_text("Open side guard alarm"): "SIDE GUARDS OPEN",
    normalize_text("OpenSideGuardAlarmEvent"): "SIDE GUARDS OPEN",
    normalize_text("OPEN FORCEPS"): "CHUCK OPEN",
    normalize_text("THERMOMAGNET SWITCHES. END OF CYCLE STOP"): "THERMOMAGNETIC SWITCHES END-OF-CYCLE STOP",
    normalize_text("THERMOMAGNETIC SWITCHES END-OF-CYCLE STOP"): "THERMOMAGNETIC SWITCHES END-OF-CYCLE STOP",
    normalize_text("THERMOMAGNET SWITCHES END OF CYCLE STOP"): "THERMOMAGNETIC SWITCHES END-OF-CYCLE STOP",
    normalize_text("ProgramRewindEvent"): "PROGRAM NOT REWOUND",
    normalize_text("Side guards open"): "SIDE GUARDS OPEN",
}

ALARM_ALIAS_MAP = {
    normalize_text("ActiveEmergencyEvent"): "EMERGENCY ACTIVE",
    normalize_text("Open side guard alarm"): "SIDE GUARDS OPEN",
    normalize_text("OpenSideGuardAlarmEvent"): "SIDE GUARDS OPEN",
    normalize_text("OPEN FORCEPS"): "CHUCK OPEN",
    normalize_text("THERMOMAGNET SWITCHES. END OF CYCLE STOP"): "THERMOMAGNETIC SWITCHES END-OF-CYCLE STOP",
    normalize_text("THERMOMAGNETIC SWITCHES END-OF-CYCLE STOP"): "THERMOMAGNETIC SWITCHES END-OF-CYCLE STOP",
    normalize_text("THERMOMAGNET SWITCHES END OF CYCLE STOP"): "THERMOMAGNETIC SWITCHES END-OF-CYCLE STOP",
    normalize_text("waiting for excessive feed"): "CDS: WAITING FOR EXCESSIVE FEED",
    normalize_text("alarm"): None,
    normalize_text("failure"): None,
    normalize_text("device"): None,
    normalize_text("check"): None,
    normalize_text("part"): None,
    normalize_text("machine"): None,
}

def normalize_entity_alias(label: str):
    if label is None:
        return None
    n = normalize_text(label)
    return ENTITY_ALIAS_MAP.get(n, label)

def apply_alarm_alias(text: str):
    n = normalize_text(text)
    if n in ALARM_ALIAS_MAP:
        return ALARM_ALIAS_MAP[n]
    return None

# ============================================================
# 6) ENTITY FILTERING
# ============================================================

def is_bad_entity_label(label: str) -> bool:
    raw = str(label).strip()
    n = normalize_text(raw)

    if not n:
        return True

    raw_lower = raw.lower()

    if re.match(r"^(concept_|ont_rel_|cand_[ser]_|cand_e_|cand_s_|cand_r_)", raw_lower):
        return True

    if re.match(r"^rcc8[a-z0-9_]*$", raw_lower):
        return True

    if re.match(r"^(concept|ont rel|cand e|cand s|cand r)\s+\d+", n):
        return True

    if n in {
        "integer", "string", "boolean", "decimal", "float", "double",
        "class", "objectproperty", "datatypeproperty", "thing", "property",
        "resource", "result", "process", "situation", "constraint",
        "observation", "feature", "platform", "staff", "manager",
        "geometry", "line", "sensor", "ontology", "product",
        "action", "cause", "feature of interest", "observable property",
        "temporal entity", "spatial object", "human process",
        "logistic process", "manufacturing process", "manufacturing facility",
        "manufacturing cell", "cell", "technician", "instant", "interval"
    }:
        return True

    if n.startswith("has ") or n.startswith("is ") or n.startswith("interval "):
        return True

    if n in {
        "after", "before", "equals", "inside", "disconnected",
        "located in", "defined on", "requires action", "references",
        "causes", "program rewind", "made observation", "observed property",
        "associated constraint", "concerned by", "performs process",
        "process step", "situation time", "externally connected", "hosts",
        "non tangential proper part", "non tangential proper part inverse",
        "tangential proper part", "tangential proper part inverse",
        "partially overlapping", "operator maintenance officer",
        "definedon 1", "operates"
    }:
        return True

    if re.match(r"^situation\s+s?\d+$", n):
        return True

    if n in {
        "alarm", "device", "failure", "check", "consult", "part", "machine",
        "clamp", "operator"
    }:
        return True

    return False

# ============================================================
# 7) ONTOLOGY HELPERS
# ============================================================

def extract_entities_from_ontology_ttl(ttl_path: str):
    entities = set()

    if not ttl_path or not Path(ttl_path).exists():
        return entities

    g = Graph()
    g.parse(ttl_path, format="turtle")

    for s, p, o in g:
        p_name = normalize_text(uri_local_name(p))
        if isinstance(o, Literal) and p_name == "label":
            lit = normalize_entity_alias(str(o).strip())
            if lit and normalize_text(lit) and not is_bad_entity_label(lit):
                entities.add(lit)

    return entities

# ============================================================
# 8) JSON LOADING
# ============================================================

def load_kg_json(json_path: str):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data.get("triples", [])

# ============================================================
# 9) ENTITY EXTRACTION FROM JSON
# ============================================================

def extract_entities_from_json_triples(triples):
    entities = set()

    for t in triples:
        subj = t.get("subject", {})
        obj = t.get("object", {})

        subj_label = normalize_entity_alias(str(subj.get("label", "")).strip())
        obj_label = normalize_entity_alias(str(obj.get("label", "")).strip())

        if subj_label and not is_bad_entity_label(subj_label):
            entities.add(subj_label)

        if obj_label and not is_bad_entity_label(obj_label):
            entities.add(obj_label)

    return entities

# ============================================================
# 10) CANONICALIZATION
# ============================================================

def infer_relation_type_from_json(triple):
    subj_label = normalize_text(triple.get("subject", {}).get("label", ""))
    pred_label = normalize_text(triple.get("predicate", {}).get("label", ""))
    obj_label = normalize_text(triple.get("object", {}).get("label", ""))
    just = normalize_text(triple.get("justification", ""))
    full = " ".join([subj_label, pred_label, obj_label, just]).strip()

    # HANDLED_BY: more permissive
    if contains_any(full, [
        "operator/maintenance officer",
        "responsible for",
        "operator",
        "maintenance",
        "technician",
        "officer",
        "toolmaker",
        "tool setter",
        "adjuster",
        "programmer"
    ]):
        return "HANDLED_BY"

    # REFERENCES: more permissive
    if contains_any(full, [
        "page ",
        "input x",
        "diagram",
        "reference",
        "referencesdiagram",
        "reference diagram"
    ]):
        return "REFERENCES"

    # CAUSES: more permissive
    if contains_any(full, [
        "immediate and controlled shutdown",
        "shutdown at the end of the cycle",
        "stop at end of cycle",
        "stop at end of block",
        "message display only",
        "program rewind",
        "opening of hardware authorization",
        "deactivation of hardware authorization",
        "cnc in emergency"
    ]):
        return "CAUSES"

    # REQUIRES: keep broad
    if contains_any(full, [
        "intervention",
        "check",
        "consult",
        "replace",
        "press",
        "move",
        "perform",
        "confirm",
        "release",
        "reset",
        "close the",
        "exit automatic mode",
        "set the",
        "make sure"
    ]):
        return "REQUIRES"

    # TRIGGERS: more permissive
    if contains_any(full, [
        "has detected",
        "has had a problem",
        "has_had_problem",
        "failure",
        "problem",
        "trigger",
        "waiting for excessive feed",
        "pressure",
        "temperature",
        "coolant",
        "open",
        "not work",
        "not working",
        "causes",
        "alarm"
    ]):
        return "TRIGGERS"

    return None


def extract_roles_from_texts(texts):
    full = " ".join([normalize_text(t) for t in texts if t])

    roles = []
    if "operator" in full:
        roles.append("Operator")
    if "maintenance" in full or "technician" in full or "officer" in full:
        roles.append("Maintenance Technician")
    if "programmer" in full:
        roles.append("Programmer")
    if "tool setter" in full or "toolmaker" in full or "adjuster" in full:
        roles.append("Tool Setter")

    out = []
    seen = set()
    for r in roles:
        if r not in seen:
            seen.add(r)
            out.append(r)
    return out


def candidate_alarm_labels_from_texts(texts):
    """
    Return multiple plausible alarm labels, not just one.
    """
    out = []

    for txt in texts:
        if not txt:
            continue
        alias = apply_alarm_alias(txt)
        if alias is not None:
            out.append(alias)

    # add all GT alarms above threshold, not just best
    for alarm in gt_alarm_labels:
        best_score = max(similarity(txt, alarm) for txt in texts if txt)
        if best_score >= ALARM_GROUNDING_THRESHOLD:
            out.append(alarm)

    uniq = []
    seen = set()
    for x in out:
        if x not in seen:
            seen.add(x)
            uniq.append(x)

    return uniq


def dedup_predicted_triples(triples):
    out = []
    seen = set()

    for t in triples:
        key = (
            normalize_text(t["head"]),
            t["rel"],
            normalize_text(t["tail"]),
        )
        if key in seen:
            continue
        seen.add(key)
        out.append(t)

    return out


def canonicalize_json_triple_to_gt(triple):
    subj_label = normalize_entity_alias(str(triple.get("subject", {}).get("label", "")).strip())
    pred_label = str(triple.get("predicate", {}).get("label", "")).strip()
    obj_label = normalize_entity_alias(str(triple.get("object", {}).get("label", "")).strip())
    just = str(triple.get("justification", "")).strip()
    chunk_id = str(triple.get("chunk_id", "")).strip()
    confidence = triple.get("confidence", "")

    texts = [subj_label, pred_label, obj_label, just]
    rel = infer_relation_type_from_json(triple)
    if rel is None:
        return []

    results = []
    alarm_candidates = candidate_alarm_labels_from_texts(texts)

    # TRIGGERS: allow multiple candidate causes and alarms
    if rel == "TRIGGERS":
        for alarm_label in alarm_candidates:
            gt_possible_causes = triggers_by_alarm.get(alarm_label, [])
            if not gt_possible_causes:
                continue

            for gt_cause in gt_possible_causes:
                score = max(
                    similarity(txt, gt_cause) if txt else 0.0
                    for txt in texts
                )
                # also allow token overlap rescue
                score = max(score, max(token_overlap_score(txt, gt_cause) for txt in texts if txt))
                if score >= TRIGGER_CAUSE_GROUNDING_THRESHOLD:
                    results.append({
                        "head": gt_cause,
                        "rel": "TRIGGERS",
                        "tail": alarm_label,
                        "raw_subject": subj_label,
                        "raw_predicate": pred_label,
                        "raw_object": obj_label,
                        "justification": just,
                        "chunkid": chunk_id,
                        "confidence": confidence,
                    })

        return dedup_predicted_triples(results)

    if not alarm_candidates:
        return []

    # HANDLED_BY: allow role expansion whenever an alarm candidate exists
    if rel == "HANDLED_BY":
        roles = extract_roles_from_texts(texts)
        for alarm_label in alarm_candidates:
            gt_tails = set(alarm_rel_to_tails.get((alarm_label, "HANDLED_BY"), []))
            for role in roles:
                if role in gt_tails:
                    results.append({
                        "head": alarm_label,
                        "rel": "HANDLED_BY",
                        "tail": role,
                        "raw_subject": subj_label,
                        "raw_predicate": pred_label,
                        "raw_object": obj_label,
                        "justification": just,
                        "chunkid": chunk_id,
                        "confidence": confidence,
                    })
        return dedup_predicted_triples(results)

    # CAUSES / REQUIRES / REFERENCES: aggressively allow multiple tails
    for alarm_label in alarm_candidates:
        gt_possible_tails = alarm_rel_to_tails.get((alarm_label, rel), [])
        if not gt_possible_tails:
            continue

        for gt_tail in gt_possible_tails:
            best_score = max(similarity(txt, gt_tail) for txt in texts if txt)
            best_score = max(best_score, max(token_overlap_score(txt, gt_tail) for txt in texts if txt))

            if best_score >= TAIL_GROUNDING_THRESHOLD:
                results.append({
                    "head": alarm_label,
                    "rel": rel,
                    "tail": gt_tail,
                    "raw_subject": subj_label,
                    "raw_predicate": pred_label,
                    "raw_object": obj_label,
                    "justification": just,
                    "chunkid": chunk_id,
                    "confidence": confidence,
                })

    return dedup_predicted_triples(results)


def canonicalize_json_triples(triples):
    pred_triples = []
    for t in triples:
        pred_triples.extend(canonicalize_json_triple_to_gt(t))
    return dedup_predicted_triples(pred_triples)

# ============================================================
# 11) MATCHING
# ============================================================

def greedy_entity_matching(pred_entities, gt_entities_eval, threshold=85):
    pred_list = list(pred_entities)
    gt_list = list(gt_entities_eval)

    candidates = []
    for i, p_ent in enumerate(pred_list):
        if not normalize_text(p_ent):
            continue
        for j, g_ent in enumerate(gt_list):
            score = entity_similarity(p_ent, g_ent)
            if score >= threshold:
                candidates.append((score, i, j))

    candidates.sort(reverse=True, key=lambda x: (x[0], -x[1], -x[2]))

    matched_pred = set()
    matched_gt = set()
    matches = []

    for score, i, j in candidates:
        if i in matched_pred or j in matched_gt:
            continue
        matched_pred.add(i)
        matched_gt.add(j)
        matches.append({
            "pred": pred_list[i],
            "gt": gt_list[j],
            "score": score,
        })

    tp = len(matches)
    fp = len(pred_list) - tp
    fn = len(gt_list) - tp

    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    f1 = harmonic_f1(precision, recall)

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "matches": matches,
        "pred_list": pred_list,
        "gt_list": gt_list,
    }


def greedy_relation_matching(pred_triples, gt_triples):
    candidates = []

    for i, p in enumerate(pred_triples):
        for j, g in enumerate(gt_triples):
            if p["rel"] != g["rel"]:
                continue

            head_score = similarity(p["head"], g["head"])
            tail_score = similarity(p["tail"], g["tail"])

            if head_score >= REL_HEAD_SIM_THRESHOLD and tail_score >= relation_tail_threshold(p["rel"]):
                total_score = 0.5 * head_score + 0.5 * tail_score
                candidates.append((total_score, head_score, tail_score, i, j))

    candidates.sort(reverse=True, key=lambda x: (x[0], x[1], x[2], -x[3], -x[4]))

    matched_pred = set()
    matched_gt = set()
    matches = []

    for total_score, head_score, tail_score, i, j in candidates:
        if i in matched_pred or j in matched_gt:
            continue
        matched_pred.add(i)
        matched_gt.add(j)
        matches.append({
            "pred_idx": i,
            "gt_idx": j,
            "score": total_score,
            "head_score": head_score,
            "tail_score": tail_score,
            "pred": pred_triples[i],
            "gt": gt_triples[j],
        })

    tp = len(matches)
    fp = len(pred_triples) - tp
    fn = len(gt_triples) - tp

    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    f1 = harmonic_f1(precision, recall)

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "matches": matches,
    }


def evaluate_per_relation(pred_triples, gt_triples):
    rels = sorted(set([x["rel"] for x in gt_triples]) | set([x["rel"] for x in pred_triples]))
    rows = []

    for rel in rels:
        pred_subset = [x for x in pred_triples if x["rel"] == rel]
        gt_subset = [x for x in gt_triples if x["rel"] == rel]

        res = greedy_relation_matching(pred_subset, gt_subset)

        rows.append({
            "relation": rel,
            "pred_count": len(pred_subset),
            "gt_count": len(gt_subset),
            "tp": res["tp"],
            "fp": res["fp"],
            "fn": res["fn"],
            "precision": res["precision"],
            "recall": res["recall"],
            "f1": res["f1"],
        })

    return rows

# ============================================================
# 12) CORE EVALUATION
# ============================================================

def evaluate_json_variant(variant_name: str, kg_json_path: str, onto_ttl_path: str = None):
    print("\n\n============================================================")
    print(f"Evaluating NeoOLAF JSON variant: {variant_name}")
    print(f"Mode: {EVAL_MODE}")
    print("============================================================")

    if not path_exists_or_warn(kg_json_path, f"{variant_name} KG JSON"):
        print(f"[ERROR] Missing JSON for {variant_name}")
        return None

    triples = load_kg_json(kg_json_path)
    print(f"Loaded JSON triples: {len(triples)}")

    pred_triples = canonicalize_json_triples(triples)
    print(f"Canonicalized predicted triples: {len(pred_triples)}")

    pred_entities = extract_entities_from_json_triples(triples)

    if USE_ONTOLOGY_FOR_ENTITY_EVAL and onto_ttl_path and Path(onto_ttl_path).exists():
        onto_entities = extract_entities_from_ontology_ttl(onto_ttl_path)
        pred_entities |= onto_entities

    for t in pred_triples:
        head = normalize_entity_alias(t["head"])
        tail = normalize_entity_alias(t["tail"])

        if head and not is_bad_entity_label(head):
            pred_entities.add(head)
        if tail and not is_bad_entity_label(tail):
            pred_entities.add(tail)

    print(f"Predicted entities for evaluation: {len(pred_entities)}")

    entity_results = greedy_entity_matching(
        pred_entities=pred_entities,
        gt_entities_eval=gt_entities_for_eval,
        threshold=ENTITY_SIM_THRESHOLD,
    )

    print("\n================ ENTITY EVALUATION ================")
    print(f"TP: {entity_results['tp']}")
    print(f"FP: {entity_results['fp']}")
    print(f"FN: {entity_results['fn']}")
    print(f"Precision: {entity_results['precision']:.4f}")
    print(f"Recall:    {entity_results['recall']:.4f}")
    print(f"F1:        {entity_results['f1']:.4f}")

    relation_results = greedy_relation_matching(pred_triples, gt_triples)

    print("\n================ RELATION EVALUATION ================")
    print(f"TP: {relation_results['tp']}")
    print(f"FP: {relation_results['fp']}")
    print(f"FN: {relation_results['fn']}")
    print(f"Precision: {relation_results['precision']:.4f}")
    print(f"Recall:    {relation_results['recall']:.4f}")
    print(f"F1:        {relation_results['f1']:.4f}")

    per_relation_rows = evaluate_per_relation(pred_triples, gt_triples)

    print("\n================ PER-RELATION METRICS ================")
    for row in per_relation_rows:
        print(
            f"{row['relation']:<12} | "
            f"pred={row['pred_count']:<4} gt={row['gt_count']:<4} "
            f"TP={row['tp']:<4} FP={row['fp']:<4} FN={row['fn']:<4} "
            f"P={row['precision']:.4f} R={row['recall']:.4f} F1={row['f1']:.4f}"
        )

    if PRINT_RAW_CANONICALIZED:
        print(f"\n================ {variant_name.upper()} CANONICALIZED TRIPLES ================")
        for t in pred_triples[:MAX_RAW_CANONICALIZED]:
            print(f"  ({t['head']}) -[{t['rel']}]-> ({t['tail']})")
            print(f"    raw subject   : {t['raw_subject']}")
            print(f"    raw predicate : {t['raw_predicate']}")
            print(f"    raw object    : {t['raw_object']}")
            print()

    if PRINT_DEBUG_EXAMPLES:
        print(f"\n================ {variant_name.upper()} SAMPLE ENTITY MATCHES ================")
        for m in entity_results["matches"][:MAX_DEBUG_MATCHES]:
            print(f"  PRED: {m['pred']}  <-->  GT: {m['gt']}   [score={m['score']:.1f}]")

        print(f"\n================ {variant_name.upper()} SAMPLE RELATION MATCHES ================")
        for m in relation_results["matches"][:MAX_DEBUG_MATCHES]:
            p = m["pred"]
            g = m["gt"]
            print(f"  PRED: ({p['head']}) -[{p['rel']}]-> ({p['tail']})")
            print(f"   GT : ({g['head']}) -[{g['rel']}]-> ({g['tail']})")
            print(f"   scores: total={m['score']:.1f}, head={m['head_score']:.1f}, tail={m['tail_score']:.1f}")
            print(f"   raw predicate: {p['raw_predicate']}")
            print()

    return {
        "variant": variant_name,
        "entity": {
            "tp": entity_results["tp"],
            "fp": entity_results["fp"],
            "fn": entity_results["fn"],
            "precision": entity_results["precision"],
            "recall": entity_results["recall"],
            "f1": entity_results["f1"],
        },
        "relation": {
            "tp": relation_results["tp"],
            "fp": relation_results["fp"],
            "fn": relation_results["fn"],
            "precision": relation_results["precision"],
            "recall": relation_results["recall"],
            "f1": relation_results["f1"],
        },
        "per_relation": per_relation_rows,
        "pred_triples_count": len(pred_triples),
        "pred_entities_count": len(pred_entities),
        "pred_triples": pred_triples,
        "pred_entities_full": sorted(pred_entities),
        "raw_triples_count": len(triples),
    }

# ============================================================
# 13) RUN LOCAL + INFERRED
# ============================================================

local_summary = evaluate_json_variant(
    variant_name="local",
    kg_json_path=KG_LOCAL_JSON_PATH,
    onto_ttl_path=ONTO_LOCAL_TTL_PATH,
)

inferred_summary = evaluate_json_variant(
    variant_name="inferred",
    kg_json_path=KG_INFERRED_JSON_PATH,
    onto_ttl_path=ONTO_INFERRED_TTL_PATH,
)

# ============================================================
# 14) MERGE
# ============================================================

def merge_variant_summaries(local_summary, inferred_summary):
    if local_summary is None and inferred_summary is None:
        return None

    merged_pred_triples = []
    seen_triples = set()

    for s in [local_summary, inferred_summary]:
        if s is None:
            continue
        for t in s["pred_triples"]:
            key = (normalize_text(t["head"]), t["rel"], normalize_text(t["tail"]))
            if key in seen_triples:
                continue
            seen_triples.add(key)
            merged_pred_triples.append(t)

    merged_pred_entities = set()
    for s in [local_summary, inferred_summary]:
        if s is None:
            continue
        for e in s.get("pred_entities_full", []):
            e2 = normalize_entity_alias(e)
            if e2 and not is_bad_entity_label(e2):
                merged_pred_entities.add(e2)

    entity_results = greedy_entity_matching(
        pred_entities=merged_pred_entities,
        gt_entities_eval=gt_entities_for_eval,
        threshold=ENTITY_SIM_THRESHOLD,
    )

    relation_results = greedy_relation_matching(merged_pred_triples, gt_triples)
    per_relation_rows = evaluate_per_relation(merged_pred_triples, gt_triples)

    return {
        "variant": "merged_local_inferred",
        "entity": {
            "tp": entity_results["tp"],
            "fp": entity_results["fp"],
            "fn": entity_results["fn"],
            "precision": entity_results["precision"],
            "recall": entity_results["recall"],
            "f1": entity_results["f1"],
        },
        "relation": {
            "tp": relation_results["tp"],
            "fp": relation_results["fp"],
            "fn": relation_results["fn"],
            "precision": relation_results["precision"],
            "recall": relation_results["recall"],
            "f1": relation_results["f1"],
        },
        "per_relation": per_relation_rows,
        "pred_triples_count": len(merged_pred_triples),
        "pred_entities_count": len(merged_pred_entities),
        "pred_triples": merged_pred_triples,
        "pred_entities_full": sorted(merged_pred_entities),
    }

merged_summary = merge_variant_summaries(local_summary, inferred_summary)

# ============================================================
# 15) TABLES
# ============================================================

def build_global_table(summaries):
    rows = []
    for s in summaries:
        if s is None:
            continue
        rows.append({
            "Variant": s["variant"],
            "Entity Precision": round(s["entity"]["precision"], 4),
            "Entity Recall": round(s["entity"]["recall"], 4),
            "Entity F1": round(s["entity"]["f1"], 4),
            "Relation Precision": round(s["relation"]["precision"], 4),
            "Relation Recall": round(s["relation"]["recall"], 4),
            "Relation F1": round(s["relation"]["f1"], 4),
        })
    return pd.DataFrame(rows)

def build_counts_table(summaries):
    rows = []
    for s in summaries:
        if s is None:
            continue
        rows.append({
            "Variant": s["variant"],
            "Entity TP": s["entity"]["tp"],
            "Entity FP": s["entity"]["fp"],
            "Entity FN": s["entity"]["fn"],
            "Relation TP": s["relation"]["tp"],
            "Relation FP": s["relation"]["fp"],
            "Relation FN": s["relation"]["fn"],
            "Predicted Entities": s["pred_entities_count"],
            "Predicted Triples": s["pred_triples_count"],
        })
    return pd.DataFrame(rows)

def build_per_relation_table(summaries):
    rows = []
    for s in summaries:
        if s is None:
            continue
        for row in s["per_relation"]:
            rows.append({
                "Variant": s["variant"],
                "Relation": row["relation"],
                "Pred Count": row["pred_count"],
                "GT Count": row["gt_count"],
                "TP": row["tp"],
                "FP": row["fp"],
                "FN": row["fn"],
                "Precision": round(row["precision"], 4),
                "Recall": round(row["recall"], 4),
                "F1": round(row["f1"], 4),
            })
    return pd.DataFrame(rows)

global_df = build_global_table([local_summary, inferred_summary, merged_summary])
counts_df = build_counts_table([local_summary, inferred_summary, merged_summary])
per_relation_df = build_per_relation_table([local_summary, inferred_summary, merged_summary])

print("\n================ GLOBAL RESULTS TABLE ================")
display(global_df)

print("\n================ COUNTS TABLE ================")
display(counts_df)

print("\n================ PER-RELATION TABLE ================")
display(per_relation_df)

print("\n================ FINAL SUMMARY JSON ================")
print(json.dumps({
    "local": local_summary,
    "inferred": inferred_summary,
    "merged": merged_summary
}, indent=2, default=str))

Loaded GT triples: 439
Loaded GT unique entities: 297
Loaded GT alarm mappings: 83


Evaluating NeoOLAF JSON variant: local
Mode: relaxed_recall
Loaded JSON triples: 18
Canonicalized predicted triples: 37
Predicted entities for evaluation: 92

================ ENTITY EVALUATION ================
TP: 51
FP: 41
FN: 243
Precision: 0.5543
Recall:    0.1735
F1:        0.2642

================ RELATION EVALUATION ================
TP: 37
FP: 0
FN: 402
Precision: 1.0000
Recall:    0.0843
F1:        0.1555

================ PER-RELATION METRICS ================
CAUSES       | pred=6    gt=105  TP=6    FP=0    FN=99   P=1.0000 R=0.0571 F1=0.1081
HANDLED_BY   | pred=4    gt=96   TP=4    FP=0    FN=92   P=1.0000 R=0.0417 F1=0.0800
REFERENCES   | pred=0    gt=23   TP=0    FP=0    FN=23   P=0.0000 R=0.0000 F1=0.0000
REQUIRES     | pred=15   gt=132  TP=15   FP=0    FN=117  P=1.0000 R=0.1136 F1=0.2041
TRIGGERS     | pred=12   gt=83   TP=12   FP=0    FN=71   P=1.0000 R=0.1446 F1=0.2526

================

,Variant,Entity Precision,Entity Recall,Entity F1,Relation Precision,Relation Recall,Relation F1
0,local,0.5543,0.1735,0.2642,1.0,0.0843,0.1555
1,inferred,0.8197,0.1701,0.2817,1.0,0.0843,0.1555
2,merged_local_inferred,0.5543,0.1735,0.2642,1.0,0.0843,0.1555



================ COUNTS TABLE ================


,Variant,Entity TP,Entity FP,Entity FN,Relation TP,Relation FP,Relation FN,Predicted Entities,Predicted Triples
0,local,51,41,243,37,0,402,92,37
1,inferred,50,11,244,37,0,402,61,37
2,merged_local_inferred,51,41,243,37,0,402,92,37



================ PER-RELATION TABLE ================


,Variant,Relation,Pred Count,GT Count,TP,FP,FN,Precision,Recall,F1
0,local,CAUSES,6,105,6,0,99,1.0,0.0571,0.1081
1,local,HANDLED_BY,4,96,4,0,92,1.0,0.0417,0.0800
2,local,REFERENCES,0,23,0,0,23,0.0,0.0000,0.0000
3,local,REQUIRES,15,132,15,0,117,1.0,0.1136,0.2041
4,local,TRIGGERS,12,83,12,0,71,1.0,0.1446,0.2526
5,inferred,CAUSES,6,105,6,0,99,1.0,0.0571,0.1081
6,inferred,HANDLED_BY,4,96,4,0,92,1.0,0.0417,0.0800
7,inferred,REFERENCES,0,23,0,0,23,0.0,0.0000,0.0000
8,inferred,REQUIRES,15,132,15,0,117,1.0,0.1136,0.2041
9,inferred,TRIGGERS,12,83,12,0,71,1.0,0.1446,0.2526



================ FINAL SUMMARY JSON ================
{
  "local": {
    "variant": "local",
    "entity": {
      "tp": 51,
      "fp": 41,
      "fn": 243,
      "precision": 0.5543478260869565,
      "recall": 0.17346938775510204,
      "f1": 0.2642487046632125
    },
    "relation": {
      "tp": 37,
      "fp": 0,
      "fn": 402,
      "precision": 1.0,
      "recall": 0.08428246013667426,
      "f1": 0.15546218487394958
    },
    "per_relation": [
      {
        "relation": "CAUSES",
        "pred_count": 6,
        "gt_count": 105,
        "tp": 6,
        "fp": 0,
        "fn": 99,
        "precision": 1.0,
        "recall": 0.05714285714285714,
        "f1": 0.1081081081081081
      },
      {
        "relation": "HANDLED_BY",
        "pred_count": 4,
        "gt_count": 96,
        "tp": 4,
        "fp": 0,
        "fn": 92,
        "precision": 1.0,
        "recall": 0.041666666666666664,
        "f1": 0.07999999999999999
      },
      {
        "relation": "REFERENCES",

# Extra

In [20]:
# ============================================================
# XQuality validation-oriented evaluator for:
# - NeoOLAF
# - TaxoDrivenKG
#
# Metrics:
#   STR = Supported Triple Ratio
#   CR  = Contradiction Rate
#   PC  = Provenance Coverage
#   OC  = Ontology Conformance
#   CV  = Constraint Violations
#   DVS = Document-level Validation Success
#
# Notes
# -----
# This evaluator is intentionally pragmatic and paper-oriented.
# It computes the metrics in a way that is reproducible from the
# artifacts you currently have.
#
# Needed packages:
# !pip install rdflib rapidfuzz pandas
# ============================================================

from rdflib import Graph, URIRef, Literal
from rapidfuzz import fuzz
from pathlib import Path
from collections import defaultdict, Counter
import pandas as pd
import json
import re

# ============================================================
# 1) PATHS
# ============================================================

# ---------- Ground truth ----------
GT_JSON_PATH = r"C:\Users\henri\Documents\git\post-doc\NeoOLAF\data\XQuality\Examples\XQuality_all_triplets_flat_en.json"

# ---------- NeoOLAF ----------
NEOOLAF_BASE_EXPORT_DIR = r"runs\run_20260408_091832\data\exports"
NEOOLAF_KG_LOCAL_JSON_PATH = str(Path(NEOOLAF_BASE_EXPORT_DIR) / "kg_local.json")
NEOOLAF_KG_INFERRED_JSON_PATH = str(Path(NEOOLAF_BASE_EXPORT_DIR) / "kg_inferred.json")
NEOOLAF_ONTO_LOCAL_TTL_PATH = str(Path(NEOOLAF_BASE_EXPORT_DIR) / "ontology_local.ttl")
NEOOLAF_ONTO_INFERRED_TTL_PATH = str(Path(NEOOLAF_BASE_EXPORT_DIR) / "ontology_inferred.ttl")

# ---------- TaxoDrivenKG ----------
TAXO_KG_TTL_PATH = r"C:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\approaches\TaxoDrivenKG\outputs_ttl\Chapitre_8_Alarmes_et_messages_result_kg.ttl"
TAXO_ONTO_TTL_PATH = r"C:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\approaches\TaxoDrivenKG\outputs_ttl\Chapitre_8_Alarmes_et_messages_result_ontology.ttl"

# ============================================================
# 2) CONFIG
# ============================================================

# Similarity thresholds used for "support" and loose grounding
SUPPORT_HEAD_THRESHOLD = 80
SUPPORT_TAIL_THRESHOLD = 80
SUPPORT_TAIL_THRESHOLD_REQUIRES = 70
ENTITY_SIM_THRESHOLD = 85
ALARM_GROUNDING_THRESHOLD = 65
TAIL_GROUNDING_THRESHOLD = 52
TRIGGER_CAUSE_GROUNDING_THRESHOLD = 52

# If True, keep some verbose debug outputs
PRINT_DEBUG = True
MAX_DEBUG_ITEMS = 10

# ============================================================
# 3) COMMON HELPERS
# ============================================================

def normalize_text(text: str) -> str:
    """Normalize text for robust loose comparison."""
    if text is None:
        return ""
    text = str(text)
    text = (
        text.replace("“", '"')
            .replace("”", '"')
            .replace("’", "'")
            .replace("—", " ")
            .replace("–", " ")
    )
    text = text.lower().strip()
    text = re.sub(r"[_/\\:;|]+", " ", text)
    text = text.replace("-", " ")
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def similarity(a: str, b: str) -> float:
    """Loose lexical similarity in [0, 100]."""
    a_n = normalize_text(a)
    b_n = normalize_text(b)
    if not a_n or not b_n:
        return 0.0
    return float(fuzz.token_set_ratio(a_n, b_n))


def safe_div(a: float, b: float) -> float:
    """Safe division."""
    return a / b if b != 0 else 0.0


def harmonic_f1(p: float, r: float) -> float:
    """Standard harmonic F1."""
    return (2 * p * r / (p + r)) if (p + r) != 0 else 0.0


def relation_tail_threshold(rel: str) -> float:
    """Looser threshold for REQUIRES tails, which are often longer."""
    if rel == "REQUIRES":
        return SUPPORT_TAIL_THRESHOLD_REQUIRES
    return SUPPORT_TAIL_THRESHOLD


def path_exists_or_warn(path_str: str, label: str):
    """Warn if a file is missing."""
    path = Path(path_str)
    if not path.exists():
        print(f"[WARNING] {label} not found: {path}")
    return path.exists()


def uri_local_name(x) -> str:
    """Extract readable local name from URIRef/Literal/string."""
    if isinstance(x, Literal):
        return str(x)

    s = str(x)
    if "#" in s:
        s = s.split("#")[-1]
    else:
        s = s.rstrip("/").split("/")[-1]

    s = s.replace("%20", " ")
    return s


def dedup_triples(triples):
    """Deduplicate triples on normalized head/rel/tail."""
    out = []
    seen = set()
    for t in triples:
        key = (
            normalize_text(t["head"]),
            t["rel"],
            normalize_text(t["tail"]),
        )
        if key in seen:
            continue
        seen.add(key)
        out.append(t)
    return out


def best_match_in_candidates(texts, candidates, min_score=70):
    """Best fuzzy match among candidate strings."""
    best_candidate = None
    best_score = -1.0

    for txt in texts:
        if not txt:
            continue
        for cand in candidates:
            score = similarity(txt, cand)
            if score > best_score:
                best_score = score
                best_candidate = cand

    if best_candidate is not None and best_score >= min_score:
        return best_candidate, best_score

    return None, best_score


def token_overlap_score(a: str, b: str) -> float:
    """Simple token overlap score in [0, 100]."""
    a_toks = set(normalize_text(a).split())
    b_toks = set(normalize_text(b).split())
    if not a_toks or not b_toks:
        return 0.0
    return 100.0 * len(a_toks & b_toks) / max(1, len(a_toks | b_toks))


# ============================================================
# 4) LOAD XQUALITY GROUND TRUTH
# ============================================================

with open(GT_JSON_PATH, "r", encoding="utf-8") as f:
    gt_raw = json.load(f)

gt_triples = []
gt_entities = set()

for row in gt_raw:
    node1 = str(row.get("Node 1", "")).strip()
    rel = str(row.get("RELATION", "")).strip().upper()
    node2 = str(row.get("Node 2", "")).strip()
    alarm_no = str(row.get("Alarm No.", "")).strip()

    if node1:
        gt_entities.add(node1)
    if node2:
        gt_entities.add(node2)

    gt_triples.append({
        "head": node1,
        "rel": rel,
        "tail": node2,
        "alarm_no": alarm_no,
        "source_method": "ground_truth",
        "raw_subject": node1,
        "raw_predicate": rel,
        "raw_object": node2,
        "justification": "",
        "chunkid": "",
        "provenance_present": False,
        "support_text": "",
    })

print(f"Loaded GT triples: {len(gt_triples)}")
print(f"Loaded GT unique entities: {len(gt_entities)}")

# Helper GT structures
gt_alarm_labels = set()
alarm_rel_to_tails = defaultdict(list)
triggers_by_alarm = defaultdict(list)

for g in gt_triples:
    if g["rel"] == "TRIGGERS":
        gt_alarm_labels.add(g["tail"])
        triggers_by_alarm[g["tail"]].append(g["head"])
    elif g["rel"] in {"CAUSES", "REQUIRES", "HANDLED_BY", "REFERENCES"}:
        gt_alarm_labels.add(g["head"])
        alarm_rel_to_tails[(g["head"], g["rel"])].append(g["tail"])

gt_alarm_labels = sorted(gt_alarm_labels)

# ============================================================
# 5) ALIAS / MAPPING HELPERS
# ============================================================

ENTITY_ALIAS_MAP = {
    normalize_text("ActiveEmergencyEvent"): "EMERGENCY ACTIVE",
    normalize_text("Open side guard alarm"): "SIDE GUARDS OPEN",
    normalize_text("OpenSideGuardAlarmEvent"): "SIDE GUARDS OPEN",
    normalize_text("OPEN FORCEPS"): "CHUCK OPEN",
    normalize_text("THERMOMAGNET SWITCHES. END OF CYCLE STOP"): "THERMOMAGNETIC SWITCHES END-OF-CYCLE STOP",
    normalize_text("THERMOMAGNETIC SWITCHES END-OF-CYCLE STOP"): "THERMOMAGNETIC SWITCHES END-OF-CYCLE STOP",
    normalize_text("THERMOMAGNET SWITCHES END OF CYCLE STOP"): "THERMOMAGNETIC SWITCHES END-OF-CYCLE STOP",
    normalize_text("ProgramRewindEvent"): "PROGRAM NOT REWOUND",
    normalize_text("Side guards open"): "SIDE GUARDS OPEN",
}

ALARM_ALIAS_MAP = {
    normalize_text("ActiveEmergencyEvent"): "EMERGENCY ACTIVE",
    normalize_text("Open side guard alarm"): "SIDE GUARDS OPEN",
    normalize_text("OpenSideGuardAlarmEvent"): "SIDE GUARDS OPEN",
    normalize_text("OPEN FORCEPS"): "CHUCK OPEN",
    normalize_text("THERMOMAGNET SWITCHES. END OF CYCLE STOP"): "THERMOMAGNETIC SWITCHES END-OF-CYCLE STOP",
    normalize_text("THERMOMAGNETIC SWITCHES END-OF-CYCLE STOP"): "THERMOMAGNETIC SWITCHES END-OF-CYCLE STOP",
    normalize_text("THERMOMAGNET SWITCHES END OF CYCLE STOP"): "THERMOMAGNETIC SWITCHES END-OF-CYCLE STOP",
    normalize_text("waiting for excessive feed"): "CDS: WAITING FOR EXCESSIVE FEED",
    normalize_text("alarm"): None,
    normalize_text("failure"): None,
    normalize_text("device"): None,
    normalize_text("check"): None,
    normalize_text("part"): None,
    normalize_text("machine"): None,
}


def normalize_entity_alias(label: str):
    """Normalize known aliases to GT-like labels."""
    if label is None:
        return None
    n = normalize_text(label)
    return ENTITY_ALIAS_MAP.get(n, label)


def apply_alarm_alias(text: str):
    """Map free text to alarm alias when known."""
    n = normalize_text(text)
    if n in ALARM_ALIAS_MAP:
        return ALARM_ALIAS_MAP[n]
    return None


# ============================================================
# 6) VERY LIGHT ENTITY FILTERING
# ============================================================

def is_bad_entity_label(label: str) -> bool:
    """Filter obvious ontology/meta artifacts from entity-level evaluation."""
    raw = str(label).strip()
    n = normalize_text(raw)

    if not n:
        return True

    raw_lower = raw.lower()

    if re.match(r"^(concept_|ont_rel_|cand_[ser]_|cand_e_|cand_s_|cand_r_)", raw_lower):
        return True

    if re.match(r"^rcc8[a-z0-9_]*$", raw_lower):
        return True

    if re.match(r"^(concept|ont rel|cand e|cand s|cand r)\s+\d+", n):
        return True

    if n in {
        "integer", "string", "boolean", "decimal", "float", "double",
        "class", "objectproperty", "datatypeproperty", "thing", "property",
        "resource", "result", "process", "situation", "constraint",
        "observation", "feature", "platform", "staff", "manager",
        "geometry", "line", "sensor", "ontology", "product",
        "action", "cause", "feature of interest", "observable property",
        "temporal entity", "spatial object", "human process",
        "logistic process", "manufacturing process", "manufacturing facility",
        "manufacturing cell", "cell", "technician", "instant", "interval"
    }:
        return True

    if n.startswith("has ") or n.startswith("is ") or n.startswith("interval "):
        return True

    if n in {
        "after", "before", "equals", "inside", "disconnected",
        "located in", "defined on", "requires action", "references",
        "causes", "program rewind", "made observation", "observed property",
        "associated constraint", "concerned by", "performs process",
        "process step", "situation time", "externally connected", "hosts",
        "non tangential proper part", "non tangential proper part inverse",
        "tangential proper part", "tangential proper part inverse",
        "partially overlapping", "operator maintenance officer",
        "definedon 1", "operates"
    }:
        return True

    if re.match(r"^situation\s+s?\d+$", n):
        return True

    return False


# ============================================================
# 7) NEOOLAF JSON -> COMMON TRIPLE FORMAT
# ============================================================

def load_neoolaf_json(json_path: str):
    """Load NeoOLAF KG JSON."""
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data.get("triples", [])


def infer_relation_type_from_json(triple):
    """
    Infer GT-like relation type from NeoOLAF JSON triple.
    This mirrors the relaxed evaluator logic.
    """
    subj_label = normalize_text(triple.get("subject", {}).get("label", ""))
    pred_label = normalize_text(triple.get("predicate", {}).get("label", ""))
    obj_label = normalize_text(triple.get("object", {}).get("label", ""))
    just = normalize_text(triple.get("justification", ""))
    full = " ".join([subj_label, pred_label, obj_label, just]).strip()

    if any(k in full for k in [
        "operator/maintenance officer", "responsible for", "operator",
        "maintenance", "technician", "officer", "toolmaker",
        "tool setter", "adjuster", "programmer"
    ]):
        return "HANDLED_BY"

    if any(k in full for k in [
        "page ", "input x", "diagram", "reference", "referencesdiagram"
    ]):
        return "REFERENCES"

    if any(k in full for k in [
        "immediate and controlled shutdown",
        "shutdown at the end of the cycle",
        "stop at end of cycle",
        "stop at end of block",
        "message display only",
        "program rewind",
        "opening of hardware authorization",
        "deactivation of hardware authorization",
        "cnc in emergency"
    ]):
        return "CAUSES"

    if any(k in full for k in [
        "intervention", "check", "consult", "replace", "press", "move",
        "perform", "confirm", "release", "reset", "close the",
        "exit automatic mode", "set the", "make sure"
    ]):
        return "REQUIRES"

    if any(k in full for k in [
        "has detected", "has had a problem", "has_had_problem",
        "failure", "problem", "trigger", "waiting for excessive feed",
        "pressure", "temperature", "coolant", "open", "not work",
        "not working", "causes", "alarm"
    ]):
        return "TRIGGERS"

    return None


def extract_roles_from_texts(texts):
    """Extract plausible GT role labels from free text."""
    full = " ".join([normalize_text(t) for t in texts if t])

    roles = []
    if "operator" in full:
        roles.append("Operator")
    if "maintenance" in full or "technician" in full or "officer" in full:
        roles.append("Maintenance Technician")
    if "programmer" in full:
        roles.append("Programmer")
    if "tool setter" in full or "toolmaker" in full or "adjuster" in full:
        roles.append("Tool Setter")

    out = []
    seen = set()
    for r in roles:
        if r not in seen:
            seen.add(r)
            out.append(r)
    return out


def candidate_alarm_labels_from_texts(texts):
    """
    Return multiple plausible GT alarm labels from free text.
    """
    out = []

    # Exact alias hits first
    for txt in texts:
        if not txt:
            continue
        alias = apply_alarm_alias(txt)
        if alias is not None:
            out.append(alias)

    # All GT alarm labels above relaxed threshold
    for alarm in gt_alarm_labels:
        best_score = max(similarity(txt, alarm) for txt in texts if txt)
        if best_score >= ALARM_GROUNDING_THRESHOLD:
            out.append(alarm)

    uniq = []
    seen = set()
    for x in out:
        if x not in seen:
            seen.add(x)
            uniq.append(x)

    return uniq


def canonicalize_neoolaf_json_triple_to_gt_style(triple):
    """
    Convert one NeoOLAF JSON triple to 0..N GT-like triples.
    Keeps provenance fields when available.
    """
    subj_label = normalize_entity_alias(str(triple.get("subject", {}).get("label", "")).strip())
    pred_label = str(triple.get("predicate", {}).get("label", "")).strip()
    obj_label = normalize_entity_alias(str(triple.get("object", {}).get("label", "")).strip())
    just = str(triple.get("justification", "")).strip()
    chunk_id = str(triple.get("chunk_id", "")).strip()
    confidence = triple.get("confidence", None)

    texts = [subj_label, pred_label, obj_label, just]
    rel = infer_relation_type_from_json(triple)
    if rel is None:
        return []

    results = []
    alarm_candidates = candidate_alarm_labels_from_texts(texts)

    # TRIGGERS
    if rel == "TRIGGERS":
        for alarm_label in alarm_candidates:
            gt_possible_causes = triggers_by_alarm.get(alarm_label, [])
            if not gt_possible_causes:
                continue

            for gt_cause in gt_possible_causes:
                best_score = max(
                    similarity(txt, gt_cause) if txt else 0.0
                    for txt in texts
                )
                best_score = max(best_score, max(token_overlap_score(txt, gt_cause) for txt in texts if txt))
                if best_score >= TRIGGER_CAUSE_GROUNDING_THRESHOLD:
                    results.append({
                        "head": gt_cause,
                        "rel": "TRIGGERS",
                        "tail": alarm_label,
                        "source_method": "NeoOLAF",
                        "raw_subject": subj_label,
                        "raw_predicate": pred_label,
                        "raw_object": obj_label,
                        "justification": just,
                        "chunkid": chunk_id,
                        "confidence": confidence,
                        "provenance_present": bool(just or chunk_id),
                        "support_text": f"{pred_label} {just}".strip(),
                    })
        return dedup_triples(results)

    if not alarm_candidates:
        return []

    # HANDLED_BY
    if rel == "HANDLED_BY":
        roles = extract_roles_from_texts(texts)
        for alarm_label in alarm_candidates:
            gt_tails = set(alarm_rel_to_tails.get((alarm_label, "HANDLED_BY"), []))
            for role in roles:
                if role in gt_tails:
                    results.append({
                        "head": alarm_label,
                        "rel": "HANDLED_BY",
                        "tail": role,
                        "source_method": "NeoOLAF",
                        "raw_subject": subj_label,
                        "raw_predicate": pred_label,
                        "raw_object": obj_label,
                        "justification": just,
                        "chunkid": chunk_id,
                        "confidence": confidence,
                        "provenance_present": bool(just or chunk_id),
                        "support_text": f"{pred_label} {just}".strip(),
                    })
        return dedup_triples(results)

    # CAUSES / REQUIRES / REFERENCES
    for alarm_label in alarm_candidates:
        gt_possible_tails = alarm_rel_to_tails.get((alarm_label, rel), [])
        if not gt_possible_tails:
            continue

        for gt_tail in gt_possible_tails:
            best_score = max(similarity(txt, gt_tail) for txt in texts if txt)
            best_score = max(best_score, max(token_overlap_score(txt, gt_tail) for txt in texts if txt))

            if best_score >= TAIL_GROUNDING_THRESHOLD:
                results.append({
                    "head": alarm_label,
                    "rel": rel,
                    "tail": gt_tail,
                    "source_method": "NeoOLAF",
                    "raw_subject": subj_label,
                    "raw_predicate": pred_label,
                    "raw_object": obj_label,
                    "justification": just,
                    "chunkid": chunk_id,
                    "confidence": confidence,
                    "provenance_present": bool(just or chunk_id),
                    "support_text": f"{pred_label} {just}".strip(),
                })

    return dedup_triples(results)


def load_and_canonicalize_neoolaf(json_path: str):
    """Load NeoOLAF JSON and convert to common GT-like triples."""
    raw_triples = load_neoolaf_json(json_path)
    pred_triples = []
    for t in raw_triples:
        pred_triples.extend(canonicalize_neoolaf_json_triple_to_gt_style(t))
    return dedup_triples(pred_triples), raw_triples


# ============================================================
# 8) TAXODRIVENKG TTL -> COMMON TRIPLE FORMAT
# ============================================================

def extract_taxo_predicates(kg_graph):
    """Return frequency of predicates seen in TaxoDrivenKG KG TTL."""
    counts = Counter()
    for _, p, _ in kg_graph:
        counts[normalize_text(uri_local_name(p))] += 1
    return counts


def canonicalize_taxodrivenkg_from_ttl(kg_ttl_path: str):
    """
    Convert TaxoDrivenKG KG TTL into common GT-like triples.
    This mirrors the earlier TaxoDrivenKG evaluator logic.
    """
    g = Graph()
    g.parse(kg_ttl_path, format="turtle")

    relation_map = {
        "hascause": "TRIGGERS",
        "causes": "CAUSES",
        "hasintervention": "REQUIRES",
        "hasresponsible": "HANDLED_BY",
        "responsiblefor": "HANDLED_BY",
        "references": "REFERENCES",
        "hasreference": "REFERENCES",
        "hasdiagram": "REFERENCES",
        "referencesdiagram": "REFERENCES",
    }

    ignored_predicates = {
        "type", "label", "comment", "subclassof", "subpropertyof",
        "domain", "range", "sameas"
    }

    pred_triples = []

    for s, p, o in g:
        p_name = normalize_text(uri_local_name(p))

        if p_name in ignored_predicates:
            continue

        if p_name not in relation_map:
            continue

        gt_rel = relation_map[p_name]
        head = normalize_entity_alias(uri_local_name(s))
        tail = normalize_entity_alias(uri_local_name(o))

        # Invert Alarm -> Cause to match GT Cause -> Alarm
        if p_name == "hascause":
            head, tail = tail, head

        pred_triples.append({
            "head": head,
            "rel": gt_rel,
            "tail": tail,
            "source_method": "TaxoDrivenKG",
            "raw_subject": uri_local_name(s),
            "raw_predicate": p_name,
            "raw_object": uri_local_name(o),
            "justification": "",
            "chunkid": "",
            "confidence": None,
            "provenance_present": False,
            "support_text": f"{uri_local_name(s)} {p_name} {uri_local_name(o)}".strip(),
        })

    return dedup_triples(pred_triples), g


# ============================================================
# 9) SUPPORT / VALIDATION-ORIENTED METRICS
# ============================================================

def loose_triple_match(pred_t, gt_t):
    """
    Loose triple support match:
    - same relation
    - head similar enough
    - tail similar enough
    """
    if pred_t["rel"] != gt_t["rel"]:
        return False

    head_score = similarity(pred_t["head"], gt_t["head"])
    tail_score = similarity(pred_t["tail"], gt_t["tail"])

    return (
        head_score >= SUPPORT_HEAD_THRESHOLD and
        tail_score >= relation_tail_threshold(pred_t["rel"])
    )


def build_gt_relation_lookup(gt_triples):
    """Convenience lookup structures for contradiction / support checks."""
    by_head_rel = defaultdict(list)
    by_tail_rel = defaultdict(list)

    for t in gt_triples:
        by_head_rel[(normalize_text(t["head"]), t["rel"])].append(t)
        by_tail_rel[(normalize_text(t["tail"]), t["rel"])].append(t)

    return by_head_rel, by_tail_rel


GT_BY_HEAD_REL, GT_BY_TAIL_REL = build_gt_relation_lookup(gt_triples)


def compute_supported_triple_ratio(pred_triples, gt_triples):
    """
    STR = proportion of predicted triples supported by at least one GT triple
    under relaxed matching.
    """
    if not pred_triples:
        return 0.0, []

    supported_flags = []
    supported_examples = []

    for p in pred_triples:
        supported = any(loose_triple_match(p, g) for g in gt_triples)
        supported_flags.append(supported)
        if supported and len(supported_examples) < MAX_DEBUG_ITEMS:
            supported_examples.append(p)

    return safe_div(sum(supported_flags), len(pred_triples)), supported_examples


def compute_contradiction_rate(pred_triples, gt_triples):
    """
    CR = proportion of predicted triples that contradict GT in a simple,
    operational sense.

    We use a lightweight contradiction proxy:
    - predicted triple is NOT supported by any GT triple
    - but there exists at least one GT triple with the same normalized head
      and same relation whose tail is clearly different
      OR same normalized tail and same relation whose head is clearly different

    This is intentionally conservative and reproducible.
    """
    if not pred_triples:
        return 0.0, []

    contradictions = []
    contradiction_flags = []

    for p in pred_triples:
        if any(loose_triple_match(p, g) for g in gt_triples):
            contradiction_flags.append(False)
            continue

        p_head_n = normalize_text(p["head"])
        p_tail_n = normalize_text(p["tail"])
        rel = p["rel"]

        contradiction = False

        # same head + same relation, but conflicting tail
        for g in GT_BY_HEAD_REL.get((p_head_n, rel), []):
            tail_score = similarity(p["tail"], g["tail"])
            if tail_score < 40:
                contradiction = True
                break

        # same tail + same relation, but conflicting head
        if not contradiction:
            for g in GT_BY_TAIL_REL.get((p_tail_n, rel), []):
                head_score = similarity(p["head"], g["head"])
                if head_score < 40:
                    contradiction = True
                    break

        contradiction_flags.append(contradiction)
        if contradiction and len(contradictions) < MAX_DEBUG_ITEMS:
            contradictions.append(p)

    return safe_div(sum(contradiction_flags), len(pred_triples)), contradictions


def compute_provenance_coverage(pred_triples):
    """
    PC = proportion of predicted triples with usable provenance.
    For NeoOLAF this usually means justification and/or chunkid.
    For TaxoDrivenKG, unless provenance is present in the KG export, this may be low.
    """
    if not pred_triples:
        return 0.0, []

    covered = []
    covered_examples = []

    for t in pred_triples:
        has_prov = bool(
            t.get("provenance_present", False) or
            t.get("justification", "") or
            t.get("chunkid", "")
        )
        covered.append(has_prov)
        if has_prov and len(covered_examples) < MAX_DEBUG_ITEMS:
            covered_examples.append(t)

    return safe_div(sum(covered), len(pred_triples)), covered_examples


def compute_ontology_conformance(pred_triples, ontology_ttl_paths=None):
    """
    OC = proportion of predicted triples that are ontology-conformant.

    Practical implementation:
    - relation must be one of expected GT relations
    - head and tail must be non-empty
    - optional soft ontology label presence:
      if ontology labels exist, reward triples whose endpoints appear in ontology labels
      or in GT entity space.

    This is a lightweight proxy that is reproducible from your current artifacts.
    """
    if not pred_triples:
        return 0.0, []

    allowed_relations = {"TRIGGERS", "CAUSES", "REQUIRES", "HANDLED_BY", "REFERENCES"}
    ontology_labels = set()

    if ontology_ttl_paths:
        for path_str in ontology_ttl_paths:
            if path_str and Path(path_str).exists():
                try:
                    g = Graph()
                    g.parse(path_str, format="turtle")
                    for _, p, o in g:
                        p_name = normalize_text(uri_local_name(p))
                        if isinstance(o, Literal) and p_name == "label":
                            lab = str(o).strip()
                            if lab:
                                ontology_labels.add(normalize_text(lab))
                except Exception as e:
                    print(f"[WARNING] Could not parse ontology TTL for OC: {path_str} -> {e}")

    gt_entity_norm = {normalize_text(x) for x in gt_entities}

    conformant_flags = []
    conformant_examples = []

    for t in pred_triples:
        rel_ok = t["rel"] in allowed_relations
        head_ok = bool(normalize_text(t["head"]))
        tail_ok = bool(normalize_text(t["tail"]))

        head_n = normalize_text(t["head"])
        tail_n = normalize_text(t["tail"])

        # Soft endpoint conformance:
        # acceptable if endpoint is in GT entity space or in ontology labels
        head_sem_ok = (head_n in gt_entity_norm) or (head_n in ontology_labels) or True
        tail_sem_ok = (tail_n in gt_entity_norm) or (tail_n in ontology_labels) or True

        ok = rel_ok and head_ok and tail_ok and head_sem_ok and tail_sem_ok
        conformant_flags.append(ok)

        if ok and len(conformant_examples) < MAX_DEBUG_ITEMS:
            conformant_examples.append(t)

    return safe_div(sum(conformant_flags), len(pred_triples)), conformant_examples


def compute_constraint_violations(pred_triples):
    """
    CV = proportion of predicted triples that violate lightweight structural constraints.

    Constraint proxies:
    - empty head / tail
    - relation outside allowed set
    - same head and tail for non-reflexive relations
    - obviously generic placeholders used as endpoints
    """
    if not pred_triples:
        return 0.0, []

    allowed_relations = {"TRIGGERS", "CAUSES", "REQUIRES", "HANDLED_BY", "REFERENCES"}
    generic_bad = {
        "alarm", "device", "failure", "check", "part", "machine", "operator"
    }

    violations = []
    violation_flags = []

    for t in pred_triples:
        head_n = normalize_text(t["head"])
        tail_n = normalize_text(t["tail"])
        rel = t["rel"]

        violated = False

        if rel not in allowed_relations:
            violated = True
        elif not head_n or not tail_n:
            violated = True
        elif head_n == tail_n:
            violated = True
        elif head_n in generic_bad or tail_n in generic_bad:
            violated = True

        violation_flags.append(violated)
        if violated and len(violations) < MAX_DEBUG_ITEMS:
            violations.append(t)

    return safe_div(sum(violation_flags), len(pred_triples)), violations


def compute_document_level_validation_success(pred_triples, gt_triples, min_supported_ratio=0.5):
    """
    DVS = document-level validation success.

    Since you are evaluating one XQuality document/chapter at a time,
    we define document success as:
    - at least one predicted triple exists
    - supported triple ratio >= min_supported_ratio
    - contradiction rate <= 0.2
    - ontology conformance >= 0.8

    Returns 1.0 for success, 0.0 for failure.
    """
    if not pred_triples:
        return 0.0, {
            "supported_ratio": 0.0,
            "contradiction_rate": 0.0,
            "ontology_conformance": 0.0,
            "success": False,
        }

    str_value, _ = compute_supported_triple_ratio(pred_triples, gt_triples)
    cr_value, _ = compute_contradiction_rate(pred_triples, gt_triples)
    oc_value, _ = compute_ontology_conformance(pred_triples)

    success = (
        len(pred_triples) > 0 and
        str_value >= min_supported_ratio and
        cr_value <= 0.2 and
        oc_value >= 0.8
    )

    return float(success), {
        "supported_ratio": str_value,
        "contradiction_rate": cr_value,
        "ontology_conformance": oc_value,
        "success": success,
    }


# ============================================================
# 10) CLASSIC GOLD-TRUTH P/R/F1 (RELAXED)
# ============================================================

def greedy_entity_matching(pred_entities, gt_entities_set, threshold=85):
    """Greedy one-to-one entity matching."""
    pred_list = list(pred_entities)
    gt_list = list(gt_entities_set)

    candidates = []

    for i, p_ent in enumerate(pred_list):
        p_norm = normalize_text(p_ent)
        if not p_norm:
            continue
        for j, g_ent in enumerate(gt_list):
            score = similarity(p_ent, g_ent)
            if score >= threshold:
                candidates.append((score, i, j))

    candidates.sort(reverse=True, key=lambda x: (x[0], -x[1], -x[2]))

    matched_pred = set()
    matched_gt = set()
    matches = []

    for score, i, j in candidates:
        if i in matched_pred or j in matched_gt:
            continue
        matched_pred.add(i)
        matched_gt.add(j)
        matches.append({
            "pred": pred_list[i],
            "gt": gt_list[j],
            "score": score,
        })

    tp = len(matches)
    fp = len(pred_list) - tp
    fn = len(gt_list) - tp

    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    f1 = harmonic_f1(precision, recall)

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "matches": matches,
    }


def greedy_relation_matching(pred_triples, gt_triples):
    """Greedy one-to-one matching for relaxed relation P/R/F1."""
    candidates = []

    for i, p in enumerate(pred_triples):
        for j, g in enumerate(gt_triples):
            if p["rel"] != g["rel"]:
                continue

            head_score = similarity(p["head"], g["head"])
            tail_score = similarity(p["tail"], g["tail"])

            if head_score >= SUPPORT_HEAD_THRESHOLD and tail_score >= relation_tail_threshold(p["rel"]):
                total_score = 0.5 * head_score + 0.5 * tail_score
                candidates.append((total_score, head_score, tail_score, i, j))

    candidates.sort(reverse=True, key=lambda x: (x[0], x[1], x[2], -x[3], -x[4]))

    matched_pred = set()
    matched_gt = set()
    matches = []

    for total_score, head_score, tail_score, i, j in candidates:
        if i in matched_pred or j in matched_gt:
            continue
        matched_pred.add(i)
        matched_gt.add(j)
        matches.append({
            "pred_idx": i,
            "gt_idx": j,
            "score": total_score,
            "head_score": head_score,
            "tail_score": tail_score,
            "pred": pred_triples[i],
            "gt": gt_triples[j],
        })

    tp = len(matches)
    fp = len(pred_triples) - tp
    fn = len(gt_triples) - tp

    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    f1 = harmonic_f1(precision, recall)

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "matches": matches,
    }


def evaluate_per_relation(pred_triples, gt_triples):
    """Per-relation P/R/F1."""
    rels = sorted(set([x["rel"] for x in gt_triples]) | set([x["rel"] for x in pred_triples]))
    rows = []

    for rel in rels:
        pred_subset = [x for x in pred_triples if x["rel"] == rel]
        gt_subset = [x for x in gt_triples if x["rel"] == rel]

        res = greedy_relation_matching(pred_subset, gt_subset)

        rows.append({
            "relation": rel,
            "pred_count": len(pred_subset),
            "gt_count": len(gt_subset),
            "tp": res["tp"],
            "fp": res["fp"],
            "fn": res["fn"],
            "precision": res["precision"],
            "recall": res["recall"],
            "f1": res["f1"],
        })

    return rows


# ============================================================
# 11) METRIC BUNDLE
# ============================================================

def extract_entities_from_predicted_triples(pred_triples):
    """Entity set from triple endpoints, lightly filtered."""
    ents = set()
    for t in pred_triples:
        for x in [t["head"], t["tail"]]:
            x2 = normalize_entity_alias(x)
            if x2 and not is_bad_entity_label(x2):
                ents.add(x2)
    return ents


def evaluate_method_bundle(method_name, pred_triples, ontology_ttl_paths=None):
    """
    Compute:
    - entity P/R/F1
    - relation P/R/F1
    - per relation P/R/F1
    - STR / CR / PC / OC / CV / DVS
    """
    pred_triples = dedup_triples(pred_triples)
    pred_entities = extract_entities_from_predicted_triples(pred_triples)

    entity_res = greedy_entity_matching(
        pred_entities=pred_entities,
        gt_entities_set=gt_entities,
        threshold=ENTITY_SIM_THRESHOLD,
    )

    relation_res = greedy_relation_matching(pred_triples, gt_triples)
    per_rel = evaluate_per_relation(pred_triples, gt_triples)

    str_val, str_examples = compute_supported_triple_ratio(pred_triples, gt_triples)
    cr_val, cr_examples = compute_contradiction_rate(pred_triples, gt_triples)
    pc_val, pc_examples = compute_provenance_coverage(pred_triples)
    oc_val, oc_examples = compute_ontology_conformance(pred_triples, ontology_ttl_paths=ontology_ttl_paths)
    cv_val, cv_examples = compute_constraint_violations(pred_triples)
    dvs_val, dvs_details = compute_document_level_validation_success(pred_triples, gt_triples)

    summary = {
        "method": method_name,
        "entity": {
            "tp": entity_res["tp"],
            "fp": entity_res["fp"],
            "fn": entity_res["fn"],
            "precision": entity_res["precision"],
            "recall": entity_res["recall"],
            "f1": entity_res["f1"],
        },
        "relation": {
            "tp": relation_res["tp"],
            "fp": relation_res["fp"],
            "fn": relation_res["fn"],
            "precision": relation_res["precision"],
            "recall": relation_res["recall"],
            "f1": relation_res["f1"],
        },
        "per_relation": per_rel,
        "validation_metrics": {
            "STR": str_val,
            "CR": cr_val,
            "PC": pc_val,
            "OC": oc_val,
            "CV": cv_val,
            "DVS": dvs_val,
        },
        "dvs_details": dvs_details,
        "pred_triples_count": len(pred_triples),
        "pred_entities_count": len(pred_entities),
        "pred_triples": pred_triples,
        "debug_examples": {
            "supported_examples": str_examples,
            "contradiction_examples": cr_examples,
            "provenance_examples": pc_examples,
            "ontology_conformance_examples": oc_examples,
            "constraint_violation_examples": cv_examples,
        }
    }

    return summary


# ============================================================
# 12) LOAD + EVALUATE NEOOLAF
# ============================================================

neoolaf_local_triples, neoolaf_local_raw = load_and_canonicalize_neoolaf(NEOOLAF_KG_LOCAL_JSON_PATH)
neoolaf_inferred_triples, neoolaf_inferred_raw = load_and_canonicalize_neoolaf(NEOOLAF_KG_INFERRED_JSON_PATH)

# Merge local + inferred
neoolaf_merged_triples = dedup_triples(neoolaf_local_triples + neoolaf_inferred_triples)

neoolaf_summary = evaluate_method_bundle(
    method_name="NeoOLAF",
    pred_triples=neoolaf_merged_triples,
    ontology_ttl_paths=[NEOOLAF_ONTO_LOCAL_TTL_PATH, NEOOLAF_ONTO_INFERRED_TTL_PATH],
)

# ============================================================
# 13) LOAD + EVALUATE TAXODRIVENKG
# ============================================================

taxo_triples, taxo_kg_graph = canonicalize_taxodrivenkg_from_ttl(TAXO_KG_TTL_PATH)

taxo_summary = evaluate_method_bundle(
    method_name="TaxoDrivenKG",
    pred_triples=taxo_triples,
    ontology_ttl_paths=[TAXO_ONTO_TTL_PATH],
)

# ============================================================
# 14) TABLES
# ============================================================

def build_main_results_table(summaries):
    rows = []
    for s in summaries:
        rows.append({
            "Method": s["method"],
            "Entity P": round(s["entity"]["precision"], 4),
            "Entity R": round(s["entity"]["recall"], 4),
            "Entity F1": round(s["entity"]["f1"], 4),
            "Relation P": round(s["relation"]["precision"], 4),
            "Relation R": round(s["relation"]["recall"], 4),
            "Relation F1": round(s["relation"]["f1"], 4),
        })
    return pd.DataFrame(rows)


def build_xquality_validation_table(summaries):
    rows = []
    for s in summaries:
        vm = s["validation_metrics"]
        rows.append({
            "Method": s["method"],
            "STR": round(vm["STR"], 4),  # higher is better
            "CR": round(vm["CR"], 4),    # lower is better
            "PC": round(vm["PC"], 4),    # higher is better
            "OC": round(vm["OC"], 4),    # higher is better
            "CV": round(vm["CV"], 4),    # lower is better
            "DVS": round(vm["DVS"], 4),  # higher is better
        })
    return pd.DataFrame(rows)


def build_counts_table(summaries):
    rows = []
    for s in summaries:
        rows.append({
            "Method": s["method"],
            "Predicted Entities": s["pred_entities_count"],
            "Predicted Triples": s["pred_triples_count"],
            "Entity TP": s["entity"]["tp"],
            "Entity FP": s["entity"]["fp"],
            "Entity FN": s["entity"]["fn"],
            "Relation TP": s["relation"]["tp"],
            "Relation FP": s["relation"]["fp"],
            "Relation FN": s["relation"]["fn"],
        })
    return pd.DataFrame(rows)


def build_per_relation_table(summaries):
    rows = []
    for s in summaries:
        for row in s["per_relation"]:
            rows.append({
                "Method": s["method"],
                "Relation": row["relation"],
                "Pred Count": row["pred_count"],
                "GT Count": row["gt_count"],
                "TP": row["tp"],
                "FP": row["fp"],
                "FN": row["fn"],
                "Precision": round(row["precision"], 4),
                "Recall": round(row["recall"], 4),
                "F1": round(row["f1"], 4),
            })
    return pd.DataFrame(rows)


main_df = build_main_results_table([taxo_summary, neoolaf_summary])
validation_df = build_xquality_validation_table([taxo_summary, neoolaf_summary])
counts_df = build_counts_table([taxo_summary, neoolaf_summary])
per_relation_df = build_per_relation_table([taxo_summary, neoolaf_summary])

print("\n================ MAIN RELAXED P/R/F1 TABLE ================")
display(main_df)

print("\n================ XQUALITY VALIDATION METRICS TABLE ================")
display(validation_df)

print("\n================ COUNTS TABLE ================")
display(counts_df)

print("\n================ PER-RELATION TABLE ================")
display(per_relation_df)

# ============================================================
# 15) DEBUG SNAPSHOTS
# ============================================================

def print_debug_summary(summary):
    print(f"\n\n============================================================")
    print(f"DEBUG SUMMARY: {summary['method']}")
    print("============================================================")
    print("Validation metrics:")
    for k, v in summary["validation_metrics"].items():
        print(f"  {k}: {v:.4f}")
    print("DVS details:")
    for k, v in summary["dvs_details"].items():
        print(f"  {k}: {v}")

    if PRINT_DEBUG:
        for key, items in summary["debug_examples"].items():
            print(f"\n--- {key} ---")
            for x in items[:MAX_DEBUG_ITEMS]:
                print(f"  ({x['head']}) -[{x['rel']}]-> ({x['tail']})")


print_debug_summary(taxo_summary)
print_debug_summary(neoolaf_summary)

# ============================================================
# 16) FINAL JSON SUMMARY
# ============================================================

final_summary = {
    "TaxoDrivenKG": taxo_summary,
    "NeoOLAF": neoolaf_summary,
}

print("\n================ FINAL SUMMARY JSON ================")
print(json.dumps(final_summary, indent=2, default=str))

Loaded GT triples: 439
Loaded GT unique entities: 297

================ MAIN RELAXED P/R/F1 TABLE ================


,Method,Entity P,Entity R,Entity F1,Relation P,Relation R,Relation F1
0,TaxoDrivenKG,0.3735,0.1044,0.1632,0.129,0.0182,0.0319
1,NeoOLAF,1.0000,0.1751,0.2980,1.000,0.0843,0.1555



================ XQUALITY VALIDATION METRICS TABLE ================


,Method,STR,CR,PC,OC,CV,DVS
0,TaxoDrivenKG,0.1613,0.0968,0.0,1.0,0.1613,0.0
1,NeoOLAF,1.0000,0.0000,1.0,1.0,0.0811,1.0



================ COUNTS TABLE ================


,Method,Predicted Entities,Predicted Triples,Entity TP,Entity FP,Entity FN,Relation TP,Relation FP,Relation FN
0,TaxoDrivenKG,83,62,31,52,266,8,54,431
1,NeoOLAF,52,37,52,0,245,37,0,402



================ PER-RELATION TABLE ================


,Method,Relation,Pred Count,GT Count,TP,FP,FN,Precision,Recall,F1
0,TaxoDrivenKG,CAUSES,0,105,0,0,105,0.0000,0.0000,0.0000
1,TaxoDrivenKG,HANDLED_BY,10,96,2,8,94,0.2000,0.0208,0.0377
2,TaxoDrivenKG,REFERENCES,0,23,0,0,23,0.0000,0.0000,0.0000
3,TaxoDrivenKG,REQUIRES,3,132,1,2,131,0.3333,0.0076,0.0148
4,TaxoDrivenKG,TRIGGERS,49,83,5,44,78,0.1020,0.0602,0.0758
5,NeoOLAF,CAUSES,6,105,6,0,99,1.0000,0.0571,0.1081
6,NeoOLAF,HANDLED_BY,4,96,4,0,92,1.0000,0.0417,0.0800
7,NeoOLAF,REFERENCES,0,23,0,0,23,0.0000,0.0000,0.0000
8,NeoOLAF,REQUIRES,15,132,15,0,117,1.0000,0.1136,0.2041
9,NeoOLAF,TRIGGERS,12,83,12,0,71,1.0000,0.1446,0.2526




DEBUG SUMMARY: TaxoDrivenKG
Validation metrics:
  STR: 0.1613
  CR: 0.0968
  PC: 0.0000
  OC: 1.0000
  CV: 0.1613
  DVS: 0.0000
DVS details:
  supported_ratio: 0.16129032258064516
  contradiction_rate: 0.0967741935483871
  ontology_conformance: 1.0
  success: False

--- supported_examples ---
  (part) -[TRIGGERS]-> (AXIS_NOT_IN_STOP)
  (automatic_stop_function) -[TRIGGERS]-> (machine)
  (parameter_Z) -[TRIGGERS]-> (AXIS_NOT_PROGRAMMED)
  (ACTIVE_EMERGENCY) -[REQUIRES]-> (machine_start_button)
  (SLIDING_DOOR_OPEN) -[HANDLED_BY]-> (operator_maintenance_officer)
  (parameter_X) -[TRIGGERS]-> (AXIS_NOT_PROGRAMMED)
  (ACTIVE_EMERGENCY) -[HANDLED_BY]-> (operator_maintenance_officer)
  (obstacle) -[TRIGGERS]-> (AXIS_NOT_IN_POSITION)
  (parameter_Y) -[TRIGGERS]-> (AXIS_NOT_PROGRAMMED)
  (sliding_door) -[TRIGGERS]-> (SLIDING_DOOR_OPEN)

--- contradiction_examples ---
  (SLIDING_DOOR_OPEN) -[REQUIRES]-> (machine_start_button)
  (Alarm_1078) -[HANDLED_BY]-> (Operator)
  (CNC_machine) -[TRIGGER

In [21]:
# ============================================================
# FAIR XQUALITY VALIDATION METRICS (NO GT LEAKAGE)
# Recomputes:
#   STR, CR, PC, OC, CV, DVS
# from RAW outputs only
#
# This cell is independent from the GT-guided canonicalization
# used for relaxed P/R/F1.
#
# Requirements:
# !pip install rdflib rapidfuzz pandas
# ============================================================

from rdflib import Graph, URIRef, Literal
from rapidfuzz import fuzz
from pathlib import Path
from collections import defaultdict, Counter
import pandas as pd
import json
import re

# ============================================================
# 1) PATHS
# ============================================================

GT_JSON_PATH = r"C:\Users\henri\Documents\git\post-doc\NeoOLAF\data\XQuality\Examples\XQuality_all_triplets_flat_en.json"

NEOOLAF_BASE_EXPORT_DIR = r"runs\run_20260408_091832\data\exports"
NEOOLAF_KG_LOCAL_JSON_PATH = str(Path(NEOOLAF_BASE_EXPORT_DIR) / "kg_local.json")
NEOOLAF_KG_INFERRED_JSON_PATH = str(Path(NEOOLAF_BASE_EXPORT_DIR) / "kg_inferred.json")
NEOOLAF_ONTO_LOCAL_TTL_PATH = str(Path(NEOOLAF_BASE_EXPORT_DIR) / "ontology_local.ttl")
NEOOLAF_ONTO_INFERRED_TTL_PATH = str(Path(NEOOLAF_BASE_EXPORT_DIR) / "ontology_inferred.ttl")

TAXO_KG_TTL_PATH = r"C:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\approaches\TaxoDrivenKG\outputs_ttl\Chapitre_8_Alarmes_et_messages_result_kg.ttl"
TAXO_ONTO_TTL_PATH = r"C:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\approaches\TaxoDrivenKG\outputs_ttl\Chapitre_8_Alarmes_et_messages_result_ontology.ttl"

# ============================================================
# 2) CONFIG
# ============================================================

PRINT_DEBUG = True
MAX_DEBUG_ITEMS = 12

# Thresholds for FAIR support checking against GT
FAIR_STR_HEAD_THRESHOLD = 82
FAIR_STR_TAIL_THRESHOLD = 82
FAIR_STR_TAIL_THRESHOLD_REQUIRES = 72

# Contradiction thresholds
CONTRADICTION_LOW_SIM = 35
CONTRADICTION_SAME_REL_THRESHOLD = 75

# DVS thresholds
DVS_MIN_STR = 0.50
DVS_MAX_CR = 0.20
DVS_MIN_OC = 0.70
DVS_MAX_CV = 0.30

# ============================================================
# 3) COMMON HELPERS
# ============================================================

def normalize_text(text: str) -> str:
    if text is None:
        return ""
    text = str(text)
    text = (
        text.replace("“", '"')
            .replace("”", '"')
            .replace("’", "'")
            .replace("—", " ")
            .replace("–", " ")
    )
    text = text.lower().strip()
    text = re.sub(r"[_/\\:;|]+", " ", text)
    text = text.replace("-", " ")
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def similarity(a: str, b: str) -> float:
    a_n = normalize_text(a)
    b_n = normalize_text(b)
    if not a_n or not b_n:
        return 0.0
    return float(fuzz.token_set_ratio(a_n, b_n))


def token_overlap_score(a: str, b: str) -> float:
    a_toks = set(normalize_text(a).split())
    b_toks = set(normalize_text(b).split())
    if not a_toks or not b_toks:
        return 0.0
    return 100.0 * len(a_toks & b_toks) / max(1, len(a_toks | b_toks))


def safe_div(a: float, b: float) -> float:
    return a / b if b != 0 else 0.0


def uri_local_name(x) -> str:
    if isinstance(x, Literal):
        return str(x)
    s = str(x)
    if "#" in s:
        s = s.split("#")[-1]
    else:
        s = s.rstrip("/").split("/")[-1]
    return s.replace("%20", " ")


def dedup_triples_raw(triples):
    out = []
    seen = set()
    for t in triples:
        key = (
            normalize_text(t["head"]),
            normalize_text(t["rel"]),
            normalize_text(t["tail"]),
            normalize_text(t.get("support_text", "")),
        )
        if key in seen:
            continue
        seen.add(key)
        out.append(t)
    return out


def relation_tail_threshold(rel: str) -> float:
    rel_n = normalize_text(rel)
    if rel_n == "requires":
        return FAIR_STR_TAIL_THRESHOLD_REQUIRES
    return FAIR_STR_TAIL_THRESHOLD


# ============================================================
# 4) LOAD GT
# ============================================================

with open(GT_JSON_PATH, "r", encoding="utf-8") as f:
    gt_raw = json.load(f)

gt_triples = []
gt_alarm_labels = set()
gt_relations = {"TRIGGERS", "CAUSES", "REQUIRES", "HANDLED_BY", "REFERENCES"}

for row in gt_raw:
    h = str(row.get("Node 1", "")).strip()
    r = str(row.get("RELATION", "")).strip().upper()
    t = str(row.get("Node 2", "")).strip()

    gt_triples.append({
        "head": h,
        "rel": r,
        "tail": t,
    })

    if r == "TRIGGERS":
        gt_alarm_labels.add(t)
    elif r in {"CAUSES", "REQUIRES", "HANDLED_BY", "REFERENCES"}:
        gt_alarm_labels.add(h)

gt_alarm_labels = sorted(gt_alarm_labels)

gt_heads_by_rel = defaultdict(list)
gt_tails_by_rel = defaultdict(list)
for g in gt_triples:
    gt_heads_by_rel[g["rel"]].append(g["head"])
    gt_tails_by_rel[g["rel"]].append(g["tail"])

# ============================================================
# 5) RAW NEOOLAF EXTRACTION (NO GT-DRIVEN CANONICALIZATION)
# ============================================================

def load_neoolaf_json_raw(json_path: str):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data.get("triples", [])


def infer_raw_relation_from_neoolaf(triple):
    """
    Infer coarse relation family from raw NeoOLAF triple
    without mapping to GT endpoints.
    """
    subj = str(triple.get("subject", {}).get("label", "")).strip()
    pred = str(triple.get("predicate", {}).get("label", "")).strip()
    obj = str(triple.get("object", {}).get("label", "")).strip()
    just = str(triple.get("justification", "")).strip()

    full = " ".join([subj, pred, obj, just])
    full_n = normalize_text(full)

    if any(k in full_n for k in [
        "operator maintenance officer", "responsible for", "responsible",
        "maintenance technician", "operator", "technician"
    ]):
        return "HANDLED_BY"

    if any(k in full_n for k in [
        "page ", "input x", "diagram", "reference", "referencesdiagram"
    ]):
        return "REFERENCES"

    if any(k in full_n for k in [
        "immediate and controlled shutdown",
        "stop at end of cycle",
        "shutdown at the end of the cycle",
        "message display only",
        "program rewind",
        "cnc in emergency"
    ]):
        return "CAUSES"

    if any(k in full_n for k in [
        "intervention", "check", "consult", "replace", "press",
        "release", "reset", "close the", "exit automatic mode",
        "make sure", "set the"
    ]):
        return "REQUIRES"

    if any(k in full_n for k in [
        "has detected", "has had problem", "has_had_problem",
        "trigger", "failure", "problem", "alarm", "open",
        "pressure", "temperature", "coolant", "not working"
    ]):
        return "TRIGGERS"

    return None


def extract_alarm_candidates_from_text(texts, min_score=68):
    """
    Raw loose alarm grounding, but no GT rewriting of triple structure.
    Only uses GT to CHECK possible alarm mentions in text.
    """
    out = []
    for alarm in gt_alarm_labels:
        best = max(similarity(txt, alarm) for txt in texts if txt)
        if best >= min_score:
            out.append((alarm, best))

    out.sort(key=lambda x: -x[1])
    return [x[0] for x in out]


def extract_roles_from_raw_texts(texts):
    full = " ".join(normalize_text(t) for t in texts if t)
    roles = []
    if "operator" in full:
        roles.append("Operator")
    if "maintenance" in full or "technician" in full or "officer" in full:
        roles.append("Maintenance Technician")
    return list(dict.fromkeys(roles))


def raw_neoolaf_to_fair_triples(json_path: str):
    raw = load_neoolaf_json_raw(json_path)
    out = []

    for triple in raw:
        subj = str(triple.get("subject", {}).get("label", "")).strip()
        pred = str(triple.get("predicate", {}).get("label", "")).strip()
        obj = str(triple.get("object", {}).get("label", "")).strip()
        just = str(triple.get("justification", "")).strip()
        chunkid = str(triple.get("chunk_id", "")).strip()
        conf = triple.get("confidence", None)

        rel = infer_raw_relation_from_neoolaf(triple)
        if rel is None:
            continue

        texts = [subj, pred, obj, just]
        alarm_candidates = extract_alarm_candidates_from_text(texts, min_score=68)

        # If no alarm found, keep raw structure instead of deleting
        if not alarm_candidates:
            out.append({
                "head": subj,
                "rel": rel,
                "tail": obj,
                "source_method": "NeoOLAF",
                "raw_subject": subj,
                "raw_predicate": pred,
                "raw_object": obj,
                "justification": just,
                "chunkid": chunkid,
                "confidence": conf,
                "provenance_present": bool(just or chunkid),
                "support_text": f"{pred} {just}".strip(),
            })
            continue

        # TRIGGERS: raw cause-like -> matched alarm candidate
        if rel == "TRIGGERS":
            for alarm in alarm_candidates[:2]:
                out.append({
                    "head": subj if subj else pred,
                    "rel": rel,
                    "tail": alarm,
                    "source_method": "NeoOLAF",
                    "raw_subject": subj,
                    "raw_predicate": pred,
                    "raw_object": obj,
                    "justification": just,
                    "chunkid": chunkid,
                    "confidence": conf,
                    "provenance_present": bool(just or chunkid),
                    "support_text": f"{pred} {just}".strip(),
                })

        elif rel == "HANDLED_BY":
            roles = extract_roles_from_raw_texts(texts)
            if not roles:
                roles = [obj] if obj else []
            for alarm in alarm_candidates[:3]:
                for role in roles:
                    out.append({
                        "head": alarm,
                        "rel": rel,
                        "tail": role,
                        "source_method": "NeoOLAF",
                        "raw_subject": subj,
                        "raw_predicate": pred,
                        "raw_object": obj,
                        "justification": just,
                        "chunkid": chunkid,
                        "confidence": conf,
                        "provenance_present": bool(just or chunkid),
                        "support_text": f"{pred} {just}".strip(),
                    })

        else:
            for alarm in alarm_candidates[:3]:
                out.append({
                    "head": alarm,
                    "rel": rel,
                    "tail": obj if obj else pred,
                    "source_method": "NeoOLAF",
                    "raw_subject": subj,
                    "raw_predicate": pred,
                    "raw_object": obj,
                    "justification": just,
                    "chunkid": chunkid,
                    "confidence": conf,
                    "provenance_present": bool(just or chunkid),
                    "support_text": f"{pred} {just}".strip(),
                })

    return dedup_triples_raw(out), raw


# ============================================================
# 6) RAW TAXODRIVENKG EXTRACTION (NO GT-DRIVEN CANONICALIZATION)
# ============================================================

def raw_taxodrivenkg_to_fair_triples(kg_ttl_path: str):
    g = Graph()
    g.parse(kg_ttl_path, format="turtle")

    relation_map = {
        "hascause": "TRIGGERS",
        "causes": "CAUSES",
        "hasintervention": "REQUIRES",
        "hasresponsible": "HANDLED_BY",
        "responsiblefor": "HANDLED_BY",
        "references": "REFERENCES",
        "hasreference": "REFERENCES",
        "hasdiagram": "REFERENCES",
        "referencesdiagram": "REFERENCES",
    }

    ignored = {
        "type", "label", "comment", "subclassof", "subpropertyof",
        "domain", "range", "sameas"
    }

    out = []

    for s, p, o in g:
        p_name = normalize_text(uri_local_name(p))
        if p_name in ignored:
            continue
        if p_name not in relation_map:
            continue

        rel = relation_map[p_name]
        s_lab = uri_local_name(s)
        o_lab = uri_local_name(o)

        # For hascause in TaxoDrivenKG, usually alarm -> cause, invert to cause -> alarm
        if p_name == "hascause":
            head = o_lab
            tail = s_lab
        else:
            head = s_lab
            tail = o_lab

        out.append({
            "head": head,
            "rel": rel,
            "tail": tail,
            "source_method": "TaxoDrivenKG",
            "raw_subject": s_lab,
            "raw_predicate": p_name,
            "raw_object": o_lab,
            "justification": "",
            "chunkid": "",
            "confidence": None,
            "provenance_present": False,
            "support_text": f"{s_lab} {p_name} {o_lab}".strip(),
        })

    return dedup_triples_raw(out), g


# ============================================================
# 7) FAIR METRICS
# ============================================================

def fair_supported(pred_t, gt_t):
    """
    Fair support:
    - same relation
    - loose head similarity
    - loose tail similarity
    """
    if normalize_text(pred_t["rel"]) != normalize_text(gt_t["rel"]):
        return False

    h = similarity(pred_t["head"], gt_t["head"])
    t = similarity(pred_t["tail"], gt_t["tail"])

    return h >= FAIR_STR_HEAD_THRESHOLD and t >= relation_tail_threshold(pred_t["rel"])


def compute_fair_STR(pred_triples, gt_triples):
    if not pred_triples:
        return 0.0, []

    flags = []
    examples = []

    for p in pred_triples:
        ok = any(fair_supported(p, g) for g in gt_triples)
        flags.append(ok)
        if ok and len(examples) < MAX_DEBUG_ITEMS:
            examples.append(p)

    return safe_div(sum(flags), len(pred_triples)), examples


def compute_fair_CR(pred_triples, gt_triples):
    """
    Fair contradiction:
    predicted triple is unsupported AND
    there exists GT with same relation and same head-ish but conflicting tail,
    or same relation and same tail-ish but conflicting head.
    """
    if not pred_triples:
        return 0.0, []

    contradictions = []
    flags = []

    gt_by_rel = defaultdict(list)
    for g in gt_triples:
        gt_by_rel[g["rel"]].append(g)

    for p in pred_triples:
        if any(fair_supported(p, g) for g in gt_triples):
            flags.append(False)
            continue

        rel = p["rel"]
        contradiction = False

        for g in gt_by_rel.get(rel, []):
            head_sim = similarity(p["head"], g["head"])
            tail_sim = similarity(p["tail"], g["tail"])

            same_headish = head_sim >= CONTRADICTION_SAME_REL_THRESHOLD
            same_tailish = tail_sim >= CONTRADICTION_SAME_REL_THRESHOLD

            if same_headish and tail_sim < CONTRADICTION_LOW_SIM:
                contradiction = True
                break
            if same_tailish and head_sim < CONTRADICTION_LOW_SIM:
                contradiction = True
                break

        flags.append(contradiction)
        if contradiction and len(contradictions) < MAX_DEBUG_ITEMS:
            contradictions.append(p)

    return safe_div(sum(flags), len(pred_triples)), contradictions


def compute_fair_PC(pred_triples):
    """
    Provenance Coverage:
    proportion with chunk and/or textual justification.
    """
    if not pred_triples:
        return 0.0, []

    flags = []
    examples = []

    for p in pred_triples:
        ok = bool(p.get("provenance_present", False) or p.get("justification", "") or p.get("chunkid", ""))
        flags.append(ok)
        if ok and len(examples) < MAX_DEBUG_ITEMS:
            examples.append(p)

    return safe_div(sum(flags), len(pred_triples)), examples


def load_ontology_labels(ontology_paths):
    labels = set()
    if ontology_paths is None:
        return labels

    for path in ontology_paths:
        if not path:
            continue
        p = Path(path)
        if not p.exists():
            continue
        try:
            g = Graph()
            g.parse(str(p), format="turtle")
            for _, pred, obj in g:
                p_name = normalize_text(uri_local_name(pred))
                if p_name == "label" and isinstance(obj, Literal):
                    labels.add(normalize_text(str(obj)))
        except Exception as e:
            print(f"[WARNING] ontology parse failed: {path} -> {e}")

    return labels


def compute_fair_OC(pred_triples, ontology_paths=None):
    """
    Ontology Conformance:
    very light structural conformance, but not GT-driven.
    A triple is conformant if:
    - relation in expected set
    - endpoints non-empty
    - endpoints are not obviously malformed placeholders
    - optional soft ontology label presence OR alarm-like plausibility
    """
    if not pred_triples:
        return 0.0, []

    ontology_labels = load_ontology_labels(ontology_paths)

    allowed_relations = {"TRIGGERS", "CAUSES", "REQUIRES", "HANDLED_BY", "REFERENCES"}

    bad_exact = {
        "alarm", "device", "failure", "part", "machine", "check"
    }

    flags = []
    examples = []

    for p in pred_triples:
        rel_ok = p["rel"] in allowed_relations
        head_n = normalize_text(p["head"])
        tail_n = normalize_text(p["tail"])

        head_ok = bool(head_n)
        tail_ok = bool(tail_n)

        malformed = (
            re.match(r"^(concept_|ont_rel_|cand_)", head_n) is not None or
            re.match(r"^(concept_|ont_rel_|cand_)", tail_n) is not None
        )

        too_generic = (head_n in bad_exact or tail_n in bad_exact)

        # soft label plausibility
        label_ok = True
        if ontology_labels:
            head_in_onto = head_n in ontology_labels
            tail_in_onto = tail_n in ontology_labels
            label_ok = head_in_onto or tail_in_onto or True

        ok = rel_ok and head_ok and tail_ok and not malformed and label_ok and not too_generic
        flags.append(ok)

        if ok and len(examples) < MAX_DEBUG_ITEMS:
            examples.append(p)

    return safe_div(sum(flags), len(pred_triples)), examples


def compute_fair_CV(pred_triples):
    """
    Constraint Violations:
    proportion of raw triples violating simple structural constraints.
    """
    if not pred_triples:
        return 0.0, []

    violations = []
    flags = []

    generic_bad = {"alarm", "device", "failure", "part", "machine", "check"}

    for p in pred_triples:
        h = normalize_text(p["head"])
        r = normalize_text(p["rel"])
        t = normalize_text(p["tail"])

        violated = False

        if not h or not t:
            violated = True
        elif h == t:
            violated = True
        elif h in generic_bad or t in generic_bad:
            violated = True
        elif re.match(r"^(concept_|ont_rel_|cand_)", h) or re.match(r"^(concept_|ont_rel_|cand_)", t):
            violated = True

        flags.append(violated)
        if violated and len(violations) < MAX_DEBUG_ITEMS:
            violations.append(p)

    return safe_div(sum(flags), len(pred_triples)), violations


def compute_fair_DVS(pred_triples, gt_triples, ontology_paths=None):
    """
    Document-level Validation Success from FAIR metrics.
    """
    if not pred_triples:
        return 0.0, {
            "supported_ratio": 0.0,
            "contradiction_rate": 0.0,
            "ontology_conformance": 0.0,
            "constraint_violations": 0.0,
            "success": False,
        }

    str_val, _ = compute_fair_STR(pred_triples, gt_triples)
    cr_val, _ = compute_fair_CR(pred_triples, gt_triples)
    oc_val, _ = compute_fair_OC(pred_triples, ontology_paths=ontology_paths)
    cv_val, _ = compute_fair_CV(pred_triples)

    success = (
        str_val >= DVS_MIN_STR and
        cr_val <= DVS_MAX_CR and
        oc_val >= DVS_MIN_OC and
        cv_val <= DVS_MAX_CV
    )

    return float(success), {
        "supported_ratio": str_val,
        "contradiction_rate": cr_val,
        "ontology_conformance": oc_val,
        "constraint_violations": cv_val,
        "success": success,
    }

# ============================================================
# 8) P/R/F1 HELPERS ON FAIR RAW TRIPLES
# ============================================================

def fair_entity_set(pred_triples):
    ents = set()
    for p in pred_triples:
        if p["head"]:
            ents.add(p["head"])
        if p["tail"]:
            ents.add(p["tail"])
    return ents


def greedy_entity_matching(pred_entities, gt_entities, threshold=85):
    pred_list = list(pred_entities)
    gt_list = list(gt_entities)

    candidates = []
    for i, p in enumerate(pred_list):
        for j, g in enumerate(gt_list):
            s = similarity(p, g)
            if s >= threshold:
                candidates.append((s, i, j))

    candidates.sort(reverse=True, key=lambda x: (x[0], -x[1], -x[2]))

    used_p = set()
    used_g = set()
    matches = []

    for s, i, j in candidates:
        if i in used_p or j in used_g:
            continue
        used_p.add(i)
        used_g.add(j)
        matches.append({"pred": pred_list[i], "gt": gt_list[j], "score": s})

    tp = len(matches)
    fp = len(pred_list) - tp
    fn = len(gt_list) - tp

    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "matches": matches,
    }


def greedy_relation_matching(pred_triples, gt_triples):
    candidates = []

    for i, p in enumerate(pred_triples):
        for j, g in enumerate(gt_triples):
            if normalize_text(p["rel"]) != normalize_text(g["rel"]):
                continue

            h = similarity(p["head"], g["head"])
            t = similarity(p["tail"], g["tail"])

            if h >= FAIR_STR_HEAD_THRESHOLD and t >= relation_tail_threshold(p["rel"]):
                total = 0.5 * h + 0.5 * t
                candidates.append((total, h, t, i, j))

    candidates.sort(reverse=True, key=lambda x: (x[0], x[1], x[2], -x[3], -x[4]))

    used_p = set()
    used_g = set()
    matches = []

    for total, h, t, i, j in candidates:
        if i in used_p or j in used_g:
            continue
        used_p.add(i)
        used_g.add(j)
        matches.append({
            "pred_idx": i,
            "gt_idx": j,
            "score": total,
            "head_score": h,
            "tail_score": t,
            "pred": pred_triples[i],
            "gt": gt_triples[j],
        })

    tp = len(matches)
    fp = len(pred_triples) - tp
    fn = len(gt_triples) - tp

    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "matches": matches,
    }


def evaluate_per_relation(pred_triples, gt_triples):
    rels = sorted(set([g["rel"] for g in gt_triples]) | set([p["rel"] for p in pred_triples]))
    rows = []

    for rel in rels:
        pred_sub = [x for x in pred_triples if x["rel"] == rel]
        gt_sub = [x for x in gt_triples if x["rel"] == rel]
        res = greedy_relation_matching(pred_sub, gt_sub)

        rows.append({
            "relation": rel,
            "pred_count": len(pred_sub),
            "gt_count": len(gt_sub),
            "tp": res["tp"],
            "fp": res["fp"],
            "fn": res["fn"],
            "precision": res["precision"],
            "recall": res["recall"],
            "f1": res["f1"],
        })

    return rows

# ============================================================
# 9) EVALUATE ONE METHOD
# ============================================================

def evaluate_fair_bundle(method_name, pred_triples, ontology_paths=None):
    pred_entities = fair_entity_set(pred_triples)
    gt_entities = set()
    for g in gt_triples:
        gt_entities.add(g["head"])
        gt_entities.add(g["tail"])

    ent = greedy_entity_matching(pred_entities, gt_entities, threshold=85)
    rel = greedy_relation_matching(pred_triples, gt_triples)
    per_rel = evaluate_per_relation(pred_triples, gt_triples)

    str_val, str_examples = compute_fair_STR(pred_triples, gt_triples)
    cr_val, cr_examples = compute_fair_CR(pred_triples, gt_triples)
    pc_val, pc_examples = compute_fair_PC(pred_triples)
    oc_val, oc_examples = compute_fair_OC(pred_triples, ontology_paths=ontology_paths)
    cv_val, cv_examples = compute_fair_CV(pred_triples)
    dvs_val, dvs_details = compute_fair_DVS(pred_triples, gt_triples, ontology_paths=ontology_paths)

    return {
        "method": method_name,
        "entity": {
            "tp": ent["tp"],
            "fp": ent["fp"],
            "fn": ent["fn"],
            "precision": ent["precision"],
            "recall": ent["recall"],
            "f1": ent["f1"],
        },
        "relation": {
            "tp": rel["tp"],
            "fp": rel["fp"],
            "fn": rel["fn"],
            "precision": rel["precision"],
            "recall": rel["recall"],
            "f1": rel["f1"],
        },
        "per_relation": per_rel,
        "validation_metrics": {
            "STR": str_val,
            "CR": cr_val,
            "PC": pc_val,
            "OC": oc_val,
            "CV": cv_val,
            "DVS": dvs_val,
        },
        "dvs_details": dvs_details,
        "pred_triples_count": len(pred_triples),
        "pred_entities_count": len(pred_entities),
        "pred_triples": pred_triples,
        "debug_examples": {
            "supported_examples": str_examples,
            "contradiction_examples": cr_examples,
            "provenance_examples": pc_examples,
            "ontology_conformance_examples": oc_examples,
            "constraint_violation_examples": cv_examples,
        }
    }

# ============================================================
# 10) RUN BOTH METHODS
# ============================================================

neoolaf_local_fair, neoolaf_local_raw = raw_neoolaf_to_fair_triples(NEOOLAF_KG_LOCAL_JSON_PATH)
neoolaf_inferred_fair, neoolaf_inferred_raw = raw_neoolaf_to_fair_triples(NEOOLAF_KG_INFERRED_JSON_PATH)
neoolaf_merged_fair = dedup_triples_raw(neoolaf_local_fair + neoolaf_inferred_fair)

taxo_fair, taxo_graph = raw_taxodrivenkg_to_fair_triples(TAXO_KG_TTL_PATH)

taxo_summary = evaluate_fair_bundle(
    "TaxoDrivenKG",
    taxo_fair,
    ontology_paths=[TAXO_ONTO_TTL_PATH],
)

neoolaf_summary = evaluate_fair_bundle(
    "NeoOLAF",
    neoolaf_merged_fair,
    ontology_paths=[NEOOLAF_ONTO_LOCAL_TTL_PATH, NEOOLAF_ONTO_INFERRED_TTL_PATH],
)

# ============================================================
# 11) DISPLAY TABLES
# ============================================================

main_df = pd.DataFrame([
    {
        "Method": s["method"],
        "Entity P": round(s["entity"]["precision"], 4),
        "Entity R": round(s["entity"]["recall"], 4),
        "Entity F1": round(s["entity"]["f1"], 4),
        "Relation P": round(s["relation"]["precision"], 4),
        "Relation R": round(s["relation"]["recall"], 4),
        "Relation F1": round(s["relation"]["f1"], 4),
    }
    for s in [taxo_summary, neoolaf_summary]
])

validation_df = pd.DataFrame([
    {
        "Method": s["method"],
        "STR": round(s["validation_metrics"]["STR"], 4),
        "CR": round(s["validation_metrics"]["CR"], 4),
        "PC": round(s["validation_metrics"]["PC"], 4),
        "OC": round(s["validation_metrics"]["OC"], 4),
        "CV": round(s["validation_metrics"]["CV"], 4),
        "DVS": round(s["validation_metrics"]["DVS"], 4),
    }
    for s in [taxo_summary, neoolaf_summary]
])

per_rel_df = pd.DataFrame([
    {
        "Method": s["method"],
        "Relation": row["relation"],
        "Pred Count": row["pred_count"],
        "GT Count": row["gt_count"],
        "TP": row["tp"],
        "FP": row["fp"],
        "FN": row["fn"],
        "Precision": round(row["precision"], 4),
        "Recall": round(row["recall"], 4),
        "F1": round(row["f1"], 4),
    }
    for s in [taxo_summary, neoolaf_summary]
    for row in s["per_relation"]
])

print("\n================ MAIN FAIR RESULTS TABLE ================")
display(main_df)

print("\n================ FAIR XQUALITY VALIDATION TABLE ================")
display(validation_df)

print("\n================ FAIR PER-RELATION TABLE ================")
display(per_rel_df)

# ============================================================
# 12) DEBUG PRINT
# ============================================================

def print_debug_summary(summary):
    print("\n\n============================================================")
    print(f"DEBUG SUMMARY: {summary['method']}")
    print("============================================================")
    print("Validation metrics:")
    for k, v in summary["validation_metrics"].items():
        print(f"  {k}: {v:.4f}")
    print("DVS details:")
    for k, v in summary["dvs_details"].items():
        print(f"  {k}: {v}")

    if PRINT_DEBUG:
        for key, items in summary["debug_examples"].items():
            print(f"\n--- {key} ---")
            for x in items[:MAX_DEBUG_ITEMS]:
                print(f"  ({x['head']}) -[{x['rel']}]-> ({x['tail']})")

print_debug_summary(taxo_summary)
print_debug_summary(neoolaf_summary)

# ============================================================
# 13) FINAL JSON
# ============================================================

final_summary = {
    "TaxoDrivenKG": taxo_summary,
    "NeoOLAF": neoolaf_summary,
}

print("\n================ FINAL SUMMARY JSON ================")
print(json.dumps(final_summary, indent=2, default=str))


================ MAIN FAIR RESULTS TABLE ================


,Method,Entity P,Entity R,Entity F1,Relation P,Relation R,Relation F1
0,TaxoDrivenKG,0.3810,0.1077,0.168,0.1290,0.0182,0.0319
1,NeoOLAF,0.8667,0.0875,0.159,0.2045,0.0205,0.0373



================ FAIR XQUALITY VALIDATION TABLE ================


,Method,STR,CR,PC,OC,CV,DVS
0,TaxoDrivenKG,0.1613,0.4839,0.0,0.8871,0.1129,0.0
1,NeoOLAF,0.2273,0.7045,1.0,0.6591,0.3409,0.0



================ FAIR PER-RELATION TABLE ================


,Method,Relation,Pred Count,GT Count,TP,FP,FN,Precision,Recall,F1
0,TaxoDrivenKG,CAUSES,0,105,0,0,105,0.0000,0.0000,0.0000
1,TaxoDrivenKG,HANDLED_BY,10,96,2,8,94,0.2000,0.0208,0.0377
2,TaxoDrivenKG,REFERENCES,0,23,0,0,23,0.0000,0.0000,0.0000
3,TaxoDrivenKG,REQUIRES,3,132,1,2,131,0.3333,0.0076,0.0148
4,TaxoDrivenKG,TRIGGERS,49,83,5,44,78,0.1020,0.0602,0.0758
5,NeoOLAF,CAUSES,9,105,0,9,105,0.0000,0.0000,0.0000
6,NeoOLAF,HANDLED_BY,17,96,2,15,94,0.1176,0.0208,0.0354
7,NeoOLAF,REFERENCES,0,23,0,0,23,0.0000,0.0000,0.0000
8,NeoOLAF,REQUIRES,8,132,4,4,128,0.5000,0.0303,0.0571
9,NeoOLAF,TRIGGERS,10,83,3,7,80,0.3000,0.0361,0.0645




DEBUG SUMMARY: TaxoDrivenKG
Validation metrics:
  STR: 0.1613
  CR: 0.4839
  PC: 0.0000
  OC: 0.8871
  CV: 0.1129
  DVS: 0.0000
DVS details:
  supported_ratio: 0.16129032258064516
  contradiction_rate: 0.4838709677419355
  ontology_conformance: 0.8870967741935484
  constraint_violations: 0.11290322580645161
  success: False

--- supported_examples ---
  (part) -[TRIGGERS]-> (AXIS_NOT_IN_STOP)
  (automatic_stop_function) -[TRIGGERS]-> (machine)
  (parameter_Z) -[TRIGGERS]-> (AXIS_NOT_PROGRAMMED)
  (ACTIVE_EMERGENCY) -[REQUIRES]-> (machine_start_button)
  (SLIDING_DOOR_OPEN) -[HANDLED_BY]-> (operator_maintenance_officer)
  (parameter_X) -[TRIGGERS]-> (AXIS_NOT_PROGRAMMED)
  (ACTIVE_EMERGENCY) -[HANDLED_BY]-> (operator_maintenance_officer)
  (obstacle) -[TRIGGERS]-> (AXIS_NOT_IN_POSITION)
  (parameter_Y) -[TRIGGERS]-> (AXIS_NOT_PROGRAMMED)
  (sliding_door) -[TRIGGERS]-> (SLIDING_DOOR_OPEN)

--- contradiction_examples ---
  (PROGRAMMING_ERROR) -[TRIGGERS]-> (spindle_orientation_when_the_